# Targeted Semantic QLoRA Fine-Tuning of Frozen Setup F1

## Research Question

**Can lightweight targeted fine-tuning improve the semantic fidelity of the unified Setup F1 reasoner for Wrong Partner detection while preserving its NORMAL, temporal-LAG, Silent Partner, structured-reasoning, and final-classification behaviour?**

This notebook begins from the **frozen Setup F1 configuration** selected after the preceding semantic-policy refinement and semantic-payload ablation.

Setup F1 is kept fixed as the reasoning architecture:

```text
Participation:
    `speaks` only

Local temporal:
    response-offset distribution only

Overlap:
    removed

Global temporal:
    all five global features retained

Semantic payload:
    coarse + focused summaries
    WITHOUT `detailed_speech_summary`

Semantic policy:
    revised branch-independent semantic policy

Reasoning format:
    Structured R1
```

The objective is no longer to change the prompt, evidence branches, or semantic payload.

Instead, this stage asks whether a **small learned adapter** can make the model use the existing F1 semantic evidence more faithfully.

---

# Motivation

Setup F1 already provides a strong compromise between predictive performance and semantic interpretability:

- strong overall classification;
- strong Wrong Partner detection;
- preserved LAG and Silent Partner performance;
- substantially improved Wrong Partner `INCOMPATIBLE` assessments relative to the earlier Structured R1 reasoner.

However, many Wrong Partner cases are still not explicitly assessed as semantically incompatible.

The fine-tuning objective is therefore targeted:

```text
Wrong Partner
        ↓
better use of existing semantic evidence
        ↓
more semantically faithful `semantic_assessment`
```

without allowing the adapter to substantially alter the other parts of the unified reasoner.

---

# Fine-Tuning Strategy

The notebook applies **lightweight QLoRA fine-tuning** to the text-language component of Qwen2.5-Omni.

The base model is loaded in 4-bit NF4 quantisation, while LoRA adapters are attached only to language-model projection layers:

```text
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj
```

Audio, vision, Talker, and token-to-wave modules are excluded because this stage operates only on the already consolidated F1 text evidence.

The executed configuration uses:

```text
LoRA rank       = 8
LoRA alpha      = 16
LoRA dropout    = 0.05
Learning rate   = 2e-5
Maximum epochs  = 3
Preservation λ  = 0.50
```

Only approximately **0.23% of the model parameters** are trainable.

---

# Targeted Training Objective

The adapter is not trained as a generic classifier.

Instead, the loss separates the semantic field from the remainder of the Structured R1 output:

$$
L_{\text{total}}
=
L_{\text{semantic}}
+
\lambda L_{\text{preservation}}
$$

## Semantic loss

`L_semantic` is applied only to the output tokens corresponding to:

```text
semantic_assessment
```

The semantic pseudo-targets are:

```text
NORMAL          → COMPATIBLE
LAG             → COMPATIBLE

WRONG PARTNER:
    Frozen F1 INCOMPATIBLE → INCOMPATIBLE
    Frozen F1 COMPATIBLE   → INCOMPATIBLE
    Frozen F1 LIMITED      → LIMITED
```

These are task-derived pseudo-targets rather than human semantic annotations.

The intention is specifically to teach Wrong Partner cases that were previously considered `COMPATIBLE` to become explicitly `INCOMPATIBLE`, while preserving genuinely uncertain `LIMITED` cases.

---

## Preservation loss

Every other assistant-output token is trained against the **frozen F1 output** as a teacher.

This preservation component is intended to retain:

- JSON structure;
- participation assessment;
- local temporal assessment;
- global temporal assessment;
- combined temporal assessment;
- decisive dimension;
- final binary prediction.

Conceptually:

```text
semantic_assessment
    → targeted adaptation

everything else
    → preserve frozen F1 behaviour
```

This makes the experiment a test of **targeted semantic adaptation**, rather than unrestricted fine-tuning of the complete reasoning output.

---

# Grouped Train / Validation / Held-Out Split

The split is performed at the `source_group_id` level.

Each original source conversation generates four related variants:

```text
NORMAL
LAG
WRONG PARTNER
SILENT PARTNER
```

All four variants from the same source conversation remain in the same split, preventing source-level leakage between training, validation, and held-out evaluation.

The notebook uses:

```text
40 source groups → training
10 source groups → validation
50 source groups → held-out evaluation
```

Gradient training uses only:

```text
NORMAL
LAG
WRONG PARTNER
```

Silent Partner cases are intentionally excluded from gradient training.

They remain in the held-out evaluation set so that Silent Partner performance can act as a preservation test for a behaviour that was never directly fine-tuned.

The final held-out evaluation therefore contains:

```text
50 NORMAL
50 LAG
50 WRONG PARTNER
50 SILENT PARTNER
= 200 cases
```

---

# What This Notebook Evaluates

After training, the best adapter is reloaded and compared directly against frozen F1 on the **same held-out cases**.

The evaluation measures both prediction quality and reasoning-field behaviour:

- binary confusion matrices;
- accuracy;
- balanced accuracy;
- macro-F1;
- per-family accuracy;
- semantic-assessment distributions;
- Wrong Partner `INCOMPATIBLE` frequency;
- preservation of non-semantic Structured R1 fields;
- exact-schema validity;
- paired prediction changes;
- exact McNemar test;
- logical-consistency violations.

The notebook then applies predefined success criteria.

The experiment is considered successful only if semantic fidelity improves **without materially damaging**:

- NORMAL performance;
- LAG performance;
- Wrong Partner classification;
- Silent Partner performance;
- semantic compatibility for NORMAL and LAG;
- structured JSON validity.

Therefore, raw accuracy alone is not sufficient to judge the fine-tuning experiment.

---

# Methodological Scope

The held-out set in this notebook is held out from **gradient training**, but it is not a fully untouched benchmark.

All 400 cases had already participated in the preceding prompt-design and ablation stages used to select Setup F1.

For this reason, the results in this notebook should be interpreted as a controlled **adapter-development evaluation** rather than the final source-disjoint generalisation test.

A later evaluation stage is required to test the frozen and fine-tuned F1 reasoners on entirely unseen source conversations.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, training configuration, split definition, adapter checkpoint, evaluation result, and diagnostic output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run once in a fresh Google Colab GPU runtime. Restart the runtime only if Colab
requests it after installation.


In [ ]:
# Install the exact runtime dependencies used by the F1 experiment
# plus PEFT for LoRA/QLoRA fine-tuning.
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece peft
!pip install -U qwen-omni-utils decord ffmpeg-python scikit-learn scipy


Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)


## 2. Mount Google Drive and load the exact artifact paths

This cell is copied from the F1 source notebook. Do not change the database,
reference, or model paths unless the Drive folder itself has moved.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Only the separate frozen NORMAL reference profiles are retained.


In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the exact 400-case consolidated database

The audit confirms 100 cases per family and four variants per source group.


In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Build the exact frozen NORMAL reference text


In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

## 6. Load the shared Structured R1 helpers

These are copied directly from the source notebook and preserve the original
prompt renderer, parser, schema, caching, and evaluation methodology.


In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The complete original Structured R1 projection is retained here
# as an internal source projection. The manual semantic-ablation helper defined later creates a deep-copied
# model-facing view that removes filtered turns, overlap and semantic summaries,
# retains participant-level `speaks`, and preserves local offsets and global features.
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


## 7. Load the improved semantic-policy workflow


In [ ]:

# ============================================================
# STRUCTURED R1 — IMPROVED SEMANTIC POLICY
# MANUAL PROMPT WORKFLOW HELPERS
#
# No automatic prompt merging or rewriting is performed.
#
# The notebook prints:
#   1. the exact targeted WRONG_PARTNER semantic prompt,
#   2. the exact full Structured R1 baseline prompt,
#      with semantics but without filtered turns/overlap.
#
# The final combined prompt must then be supplied manually.
#
# Model-facing evidence is fixed to:
#   - participant `speaks` only,
#   - complete local offset distribution,
#   - all five global temporal features,
#   - complete coarse + focused semantic summaries,
#   - no filtered turns,
#   - no overlap.
# ============================================================

from collections import Counter
from IPython.display import display

import copy
import json
import os
import re
import time


IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME = (
    "manual_ablation_l1_no_overlap_evidence_speaks_only"
)

IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH = (
    OUT_DIR
    / IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
    / "prompt_template.txt"
)

IMPROVED_SEMANTIC_EXPERIMENT_VERSION = (
    "structured_r1_improved_semantic_policy_"
    "speaks_offsets_all_global_no_overlap"
)

IMPROVED_SEMANTIC_EXPERIMENT_TITLE = (
    "Structured R1 — Improved Semantic Policy — "
    "Speaks + Offsets + All Global + Coarse/Focused — No Overlap"
)

IMPROVED_SEMANTIC_ABLATION_ID = (
    "R1_IMPROVED_SEMANTIC_POLICY_"
    "SPEAKS_OFFSETS_ALL_GLOBAL_NO_OVERLAP"
)

IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS = set()


LOCAL_OVERLAP_FIELDS_IMPROVED = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
]

LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]

GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]

assert (
    LOCAL_OVERLAP_FIELDS_IMPROVED
    + LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    == LOCAL_FEATURE_FIELDS
)


# ============================================================
# EXACT TARGETED WRONG-PARTNER PROMPT
#
# Extracted from the retained coarse + full-focused semantic-only
# validation experiment. It is printed for manual comparison only.
# ============================================================

TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE = 'You are determining whether two people belong to the SAME real\n120-second dyadic conversation.\n\nYou are given semantic summaries from two synchronized 60-second\nsegments for Participant A and Participant B.\n\nYou must use ONLY the supplied semantic fields.\n\nFor each participant segment, the input contains:\n\nCOARSE SUMMARY:\n- speech_content_summary\n- apparent_topic\n\nFOCUSED SUMMARY:\n- detailed_speech_summary\n- main_topic\n- secondary_topics\n- key_semantic_details\n- summary_specificity\n- unclear_content\n- confidence\n\nThe coarse and focused fields are two independently generated semantic\ndescriptions of the SAME participant segment. Interpret them together.\nDo not treat differences between a participant\'s own coarse and focused\nsummaries as evidence of wrong partner.\n\nDo not use visual engagement, interaction style, speaking amount,\nlistening behavior, or any other information.\n\nThe summaries were independently generated and may sometimes be broad,\nimperfect, or uncertain.\n\nIMPORTANT INTERPRETATION\n\nNORMAL:\n- The two participants\' content can plausibly belong to the same conversation.\n- They may discuss different aspects of a shared subject.\n- One participant\'s content may plausibly respond to, elaborate on, or provide\n  context for the other participant\'s content.\n- A conversation may naturally change topic between Segment 0 and Segment 1.\n- Exact word or topic-label matching is not required.\n\nWRONG_PARTNER:\n- The participants repeatedly discuss unrelated concrete subjects.\n- There is no plausible shared conversational context across the 120 seconds.\n- Both synchronized segments show semantic mismatch, or one segment shows a\n  strong concrete mismatch while the other provides no credible compatibility.\n- The fact that both participants discuss personal experiences, preferences,\n  opinions, daily life, products, or general topics is NOT by itself evidence\n  that they belong to the same conversation.\n\nGENERIC OR WEAK SUMMARIES\n\nBroad labels such as:\n- personal experiences\n- personal preferences\n- general discussion\n- daily life\n- opinions\n- lifestyle\n- personal well-being\n\nmust not be treated as evidence of compatibility unless the concrete content\nalso provides a plausible semantic connection.\n\nIf one segment is vague or generic, treat that segment as\nINSUFFICIENT_EVIDENCE rather than as evidence for NORMAL.\n\nEvaluate the complete 120-second pattern. Do not use a simple vote between\nthe two segments.\n\nReturn ONLY valid JSON with exactly this schema:\n\n{{\n  "label": "NORMAL or ANOMALOUS",\n  "anomaly_type": "none or wrong_partner",\n  "confidence": 0.0,\n  "segment_0_assessment": "compatible / mismatch / insufficient_evidence",\n  "segment_1_assessment": "compatible / mismatch / insufficient_evidence",\n  "cross_segment_assessment": "short assessment of the complete 120-second semantic relationship",\n  "reasoning": "brief evidence-based explanation"\n}}\n\nSEMANTIC INPUT:\n\n{semantic_input}'


WP_DATABASE_SEGMENTS = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]

WP_COARSE_FIELDS_EXACT = [
    "speech_content_summary",
    "apparent_topic",
]

WP_FOCUSED_FIELDS_EXACT = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


def build_exact_wp_semantic_input(case):
    combined = {
        "participant_A": {},
        "participant_B": {},
    }

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_semantics = case[
            "semantic_summaries"
        ][role]

        for segment_name in WP_DATABASE_SEGMENTS:
            segment_record = participant_semantics[
                segment_name
            ]

            coarse = select_exact_fields(
                segment_record[
                    "coarse_summary"
                ],
                WP_COARSE_FIELDS_EXACT,
                f"{role}/{segment_name}/coarse_summary",
            )

            focused = select_exact_fields(
                segment_record[
                    "focused_summary"
                ],
                WP_FOCUSED_FIELDS_EXACT,
                f"{role}/{segment_name}/focused_summary",
            )

            assert "speaks" not in focused

            combined[role][segment_name] = {
                **coarse,
                "focused_summary": focused,
            }

    return combined


def print_targeted_wrong_partner_prompt(
    *,
    case_index=0,
    prefer_wrong_partner=True,
):
    wp_cases = sorted(
        [
            case
            for case in consolidation_cases
            if get_case_family(case) in {
                "normal",
                "wrong_partner",
            }
        ],
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(wp_cases) == 200

    family_counts = Counter(
        get_case_family(case)
        for case in wp_cases
    )

    assert family_counts == {
        "normal": 100,
        "wrong_partner": 100,
    }

    if prefer_wrong_partner:
        candidate_cases = [
            case
            for case in wp_cases
            if get_case_family(case)
            == "wrong_partner"
        ]
    else:
        candidate_cases = wp_cases

    assert 0 <= case_index < len(candidate_cases)

    case = candidate_cases[
        case_index
    ]

    semantic_input = (
        build_exact_wp_semantic_input(
            case
        )
    )

    rendered_prompt = (
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        .format(
            semantic_input=json.dumps(
                semantic_input,
                indent=2,
                ensure_ascii=False,
            )
        )
    )

    print("=" * 100)
    print(
        "EXACT TARGETED NORMAL vs WRONG_PARTNER PROMPT"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Case metadata is printed outside the model prompt only."
    )
    print(
        "Temporal evidence supplied:",
        False,
    )
    print(
        "Participant speaks supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + full focused",
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED TARGETED PROMPT")
    print("=" * 100)
    print(
        rendered_prompt
    )

    return {
        "case": case,
        "semantic_input": semantic_input,
        "prompt": rendered_prompt,
    }


# ============================================================
# LOAD THE FROZEN FULL STRUCTURED R1 BASELINE PROMPT
#
# Exact L1 No-Overlap baseline:
# speaks + offsets + all global + coarse/focused semantics.
# ============================================================

def load_full_r1_no_overlap_prompt_template():
    assert (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .exists()
    ), (
        "The validated full Structured R1 L1 No-Overlap prompt "
        "was not found:\n"
        f"{IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH}\n\n"
        "Run the L1 configuration/inspection cell in the local "
        "temporal ablation notebook first so prompt_template.txt "
        "is available."
    )

    template = (
        IMPROVED_SEMANTIC_BASELINE_PROMPT_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    required_markers = [
        "Whether each participant speaks",
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for marker in required_markers:
        assert marker in template, (
            "The saved full Structured R1 prompt is missing: "
            f"{marker}"
        )

    forbidden_markers = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    template_lower = template.lower()

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content exists in the "
            f"saved baseline prompt: {marker}"
        )

    return template


# ============================================================
# FIXED MODEL-FACING INPUT
# ============================================================

def build_improved_semantic_model_input(case):
    payload = copy.deepcopy(
        build_binary_model_input(
            case
        )
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_payload = payload[
            role
        ]

        assert "speaks" in participant_payload
        assert "filtered_turns" in participant_payload

        participant_payload.pop(
            "filtered_turns"
        )

        assert set(
            participant_payload
        ) == {
            "speaks"
        }

    local_features = payload[
        "local_temporal_features"
    ]

    for field in (
        LOCAL_OVERLAP_FIELDS_IMPROVED
    ):
        local_features.pop(
            field
        )

    assert set(
        local_features
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED
    )

    assert (
        "semantic_summaries"
        in payload
    )

    semantic_keys = (
        collect_nested_keys_reasoning(
            payload[
                "semantic_summaries"
            ]
        )
    )

    for required_key in [
        "speech_content_summary",
        "apparent_topic",
        "detailed_speech_summary",
        "main_topic",
        "secondary_topics",
        "key_semantic_details",
        "summary_specificity",
        "unclear_content",
        "confidence",
    ]:
        assert any(
            key.endswith(
                required_key
            )
            for key in semantic_keys
        ), (
            "Missing semantic field: "
            f"{required_key}"
        )

    return payload


def render_full_r1_prompt_template(
    prompt_template,
    payload,
):
    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        )

    return prompt


def print_full_r1_no_overlap_prompt(
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    template = (
        load_full_r1_no_overlap_prompt_template()
    )

    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_full_r1_prompt_template(
            template,
            payload,
        )
    )

    export_dir = (
        OUT_DIR
        / "improved_semantic_policy_manual_prompts"
    )

    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    export_path = (
        export_dir
        / "full_structured_r1_no_overlap_rendered_prompt.txt"
    )

    export_path.write_text(
        prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        "EXACT FULL STRUCTURED R1 BASELINE PROMPT"
    )
    print(
        "Speaks + Offsets + All Global + Coarse/Focused"
    )
    print(
        "No Filtered Turns · No Overlap"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap evidence supplied:",
        False,
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print("\nEXACT MODEL-FACING PAYLOAD")
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED STRUCTURED R1 PROMPT")
    print("=" * 100)
    print(
        prompt
    )
    print("\nSaved rendered prompt:", export_path)

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
        "template": template,
        "export_path": export_path,
    }


# ============================================================
# MANUAL COMBINED-PROMPT RENDERER
# ============================================================

def render_manual_improved_semantic_prompt(
    prompt_template,
    payload,
):
    assert isinstance(
        prompt_template,
        str,
    )

    assert (
        prompt_template.strip()
    )

    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = (
        prompt.lower()
    )

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field leaked into prompt: "
            f"{forbidden_key}"
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert (
            forbidden_marker
            not in prompt_lower
        ), (
            "Forbidden turns/overlap marker: "
            f"{forbidden_marker}"
        )

    return prompt


# ============================================================
# MANUAL PROMPT VALIDATOR
# ============================================================

def validate_manual_improved_semantic_prompt(
    prompt_template,
):
    assert isinstance(
        prompt_template,
        str,
    ), (
        "Set R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE "
        "to one complete prompt string."
    )

    template = (
        prompt_template.strip()
    )

    assert template

    required_placeholders = [
        "{duration_seconds}",
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for placeholder in required_placeholders:
        assert placeholder in template, (
            "Missing placeholder: "
            f"{placeholder}"
        )

    required_sections = [
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "FINAL COMBINED DECISION",
        "CURRENT CASE",
        "STRUCTURED REASONING OUTPUT",
        "OUTPUT",
    ]

    for marker in required_sections:
        assert marker in template, (
            "Missing required section: "
            f"{marker}"
        )

    required_schema_markers = [
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"semantic_assessment"',
        '"decisive_dimension"',
        '"label"',
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ]

    for marker in required_schema_markers:
        assert marker in template, (
            "Missing structured-output marker: "
            f"{marker}"
        )

    semantic_policy_markers = [
        "plausible",
        "concrete",
        "insufficient",
        "generic",
        "same conversation",
    ]

    template_lower = (
        template.lower()
    )

    for marker in semantic_policy_markers:
        assert marker in template_lower, (
            "The manual semantic policy appears incomplete. "
            f"Missing concept: {marker}"
        )

    forbidden_markers = [
        "filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content remains: "
            f"{marker}"
        )

    return template


def build_manual_improved_semantic_prompt(
    case,
    config,
):
    payload = (
        build_improved_semantic_model_input(
            case
        )
    )

    prompt = (
        render_manual_improved_semantic_prompt(
            config[
                "reasoning_prompt_template"
            ],
            payload,
        )
    )

    return (
        prompt,
        payload,
    )


# ============================================================
# PREPARE CONFIG
# ============================================================

def prepare_manual_improved_semantic_experiment(
    *,
    manual_prompt_template,
):
    prompt_template = (
        validate_manual_improved_semantic_prompt(
            manual_prompt_template
        )
    )

    baseline_template = (
        load_full_r1_no_overlap_prompt_template()
    )

    experiment_dir = (
        OUT_DIR
        / IMPROVED_SEMANTIC_EXPERIMENT_VERSION
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),
        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),
        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "baseline_prompt_copy": (
            experiment_dir
            / "baseline_prompt_template.txt"
        ),
        "targeted_wp_prompt_copy": (
            experiment_dir
            / "targeted_wrong_partner_prompt_template.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    reasoning_prompt_sha256 = (
        sha256_text(
            prompt_template
        )
    )

    baseline_prompt_sha256 = (
        sha256_text(
            baseline_template
        )
    )

    targeted_wp_prompt_sha256 = (
        sha256_text(
            TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE
        )
    )

    normal_reference_sha256 = (
        sha256_text(
            normal_base_reference_text
            + "\n"
            + normal_global_reference_text
        )
    )

    paths[
        "prompt_template"
    ].write_text(
        prompt_template,
        encoding="utf-8",
    )

    paths[
        "baseline_prompt_copy"
    ].write_text(
        baseline_template,
        encoding="utf-8",
    )

    paths[
        "targeted_wp_prompt_copy"
    ].write_text(
        TARGETED_WRONG_PARTNER_PROMPT_TEMPLATE,
        encoding="utf-8",
    )

    example_prompt, example_payload = (
        build_manual_improved_semantic_prompt(
            consolidation_cases[0],
            {
                "reasoning_prompt_template": (
                    prompt_template
                )
            },
        )
    )

    manifest = {
        "experiment_version": (
            IMPROVED_SEMANTIC_EXPERIMENT_VERSION
        ),
        "experiment_title": (
            IMPROVED_SEMANTIC_EXPERIMENT_TITLE
        ),
        "ablation_id": (
            IMPROVED_SEMANTIC_ABLATION_ID
        ),
        "source_experiment": (
            IMPROVED_SEMANTIC_BASELINE_EXPERIMENT_NAME
        ),
        "model_id": MODEL_ID,
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),
        "baseline_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "targeted_wp_prompt_sha256": (
            targeted_wp_prompt_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "participant_model_input": (
            "speaks_only"
        ),
        "local_model_input": (
            "complete_offset_distribution_only"
        ),
        "global_model_input": (
            "all_five_global_features"
        ),
        "temporal_profiles_used": (
            "frozen_NORMAL_only"
        ),
        "assessment_policy": (
            "structured_R1_with_targeted_semantic_policy"
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": (
            experiment_dir
        ),
        "reasoning_prompt_template": (
            prompt_template
        ),
        "paths": paths,
    }

    print("=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — CONFIGURATION READY"
    )
    print("=" * 100)
    print(
        "Experiment version:",
        config[
            "experiment_version"
        ],
    )
    print(
        "Cases:",
        len(
            consolidation_cases
        ),
    )
    print(
        "Semantic input:",
        config[
            "semantic_input"
        ],
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print(
        "Prompt SHA256:",
        reasoning_prompt_sha256,
    )
    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )
    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )
    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )

    return config


# ============================================================
# REQUIRED INSPECTION
# ============================================================

def inspect_manual_improved_semantic_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert (
        0
        <= case_index
        < len(
            ordered_cases
        )
    )

    case = ordered_cases[
        case_index
    ]

    prompt, payload = (
        build_manual_improved_semantic_prompt(
            case,
            config,
        )
    )

    print("=" * 100)
    print(
        "MANUAL PROMPT INSPECTION — "
        "STRUCTURED R1 IMPROVED SEMANTIC POLICY"
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print(
        "Participant evidence supplied:",
        "speaks only",
    )
    print(
        "Local evidence supplied:",
        "complete offset distribution only",
    )
    print(
        "Global evidence supplied:",
        "all five global temporal features",
    )
    print(
        "Semantic evidence supplied:",
        "coarse + focused",
    )
    print(
        "Filtered turns supplied:",
        False,
    )
    print(
        "Overlap supplied:",
        False,
    )
    print("\n" + "=" * 100)
    print("EXACT MODEL-FACING PAYLOAD")
    print("=" * 100)
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )
    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)
    print(
        prompt
    )

    IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS.add(
        config[
            "experiment_version"
        ]
    )

    print(
        "\nInspection unlocked:",
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS,
    )

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
    }


# ============================================================
# CACHE
# ============================================================

def create_improved_semantic_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": (
            config[
                "model_id"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "baseline_prompt_sha256": (
            config[
                "baseline_prompt_sha256"
            ]
        ),
        "targeted_wp_prompt_sha256": (
            config[
                "targeted_wp_prompt_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": (
            config[
                "semantic_input"
            ]
        ),
        "focused_summaries_used": True,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


# ============================================================
# INFERENCE
# ============================================================

def run_manual_improved_semantic_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in IMPROVED_SEMANTIC_INSPECTED_EXPERIMENTS
    ), (
        "Run the exact manual prompt-inspection cell "
        "before inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache = (
        create_improved_semantic_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "source_experiment",
            "model_id",
            "reasoning_prompt_sha256",
            "baseline_prompt_sha256",
            "targeted_wp_prompt_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "filtered_turns_supplied",
            "overlap_supplied",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[
                    key
                ]
                == expected_cache[
                    key
                ]
            ), (
                "Cache mismatch for "
                f"{key}"
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = expected_cache

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(
        ordered_cases
    ) == 400

    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, payload = (
            build_manual_improved_semantic_prompt(
                case,
                config,
            )
        )

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        prompt_sha256 = (
            sha256_text(
                prompt
            )
        )

        payload_sha256 = (
            sha256_text(
                canonical_json(
                    payload
                )
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )
            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == payload_sha256
            )
            continue

        started = (
            time.perf_counter()
        )

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": (
                get_case_family(
                    case
                )
            ),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "prompt_sha256": (
                prompt_sha256
            ),
            "input_payload_sha256": (
                payload_sha256
            ),
            "semantic_input": (
                "coarse_and_focused"
            ),
            "focused_summaries_used": True,
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": (
                raw_output
            ),
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),
            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),
            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),
            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),
            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),
            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": (
                generation_error
            ),
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 100)
    print(
        "IMPROVED SEMANTIC POLICY — INFERENCE COMPLETE"
    )
    print("=" * 100)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


## 8. Reconstruct the exact improved semantic prompt

Only the semantic section differs from the earlier Structured R1 baseline,
exactly as in the source notebook.


In [ ]:
# STEP 3 — MANUALLY DEFINE THE COMPLETE COMBINED PROMPT
#
# Do not generate this automatically.
#
# Start from the exact full Structured R1 prompt printed above.
# Keep its participation, local temporal, global temporal,
# final-decision and structured-output logic fixed.
#
# Replace only the semantic reasoning policy with a carefully
# integrated version of the targeted wrong-partner logic.
#
# The final prompt must retain all placeholders:
#   {duration_seconds}
#   {participant_A_speaks}
#   {participant_B_speaks}
#   {local_temporal_features}
#   {global_shift_features}
#   {semantic_summaries}

# ============================================================
# STEP 3 — MANUALLY DEFINE THE IMPROVED SEMANTIC R1 PROMPT
#
# Surgical intervention:
# - keep the complete Structured R1 baseline unchanged
# - replace only the SEMANTIC EVIDENCE policy
# ============================================================

import difflib


# Load the exact full Structured R1 baseline template:
# - speaks only
# - complete local offset distribution
# - all five global features
# - coarse + focused semantics
# - no filtered turns
# - no overlap
ORIGINAL_FULL_R1_PROMPT_TEMPLATE = (
    load_full_r1_no_overlap_prompt_template()
)


SEMANTIC_SECTION_HEADER = """
============================================================
SEMANTIC EVIDENCE
============================================================
""".strip()


FINAL_DECISION_SECTION_HEADER = """
============================================================
FINAL COMBINED DECISION
============================================================
""".strip()


# ============================================================
# VERIFY THAT THE TWO SECTION BOUNDARIES ARE UNIQUE
# ============================================================

assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        SEMANTIC_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one SEMANTIC EVIDENCE section."
)


assert (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.count(
        FINAL_DECISION_SECTION_HEADER
    )
    == 1
), (
    "Expected exactly one FINAL COMBINED DECISION section."
)


(
    PROMPT_BEFORE_SEMANTIC,
    semantic_header_found,
    PROMPT_FROM_SEMANTIC_ONWARD,
) = ORIGINAL_FULL_R1_PROMPT_TEMPLATE.partition(
    SEMANTIC_SECTION_HEADER
)


assert semantic_header_found == SEMANTIC_SECTION_HEADER


(
    ORIGINAL_SEMANTIC_POLICY_BODY,
    final_decision_header_found,
    PROMPT_AFTER_FINAL_DECISION_HEADER,
) = PROMPT_FROM_SEMANTIC_ONWARD.partition(
    FINAL_DECISION_SECTION_HEADER
)


assert (
    final_decision_header_found
    == FINAL_DECISION_SECTION_HEADER
)


ORIGINAL_SEMANTIC_POLICY_BODY = (
    ORIGINAL_SEMANTIC_POLICY_BODY.strip()
)


# ============================================================
# IMPROVED SEMANTIC POLICY
#
# Adapted from the successful targeted wrong-partner pipeline.
#
# Crucially:
# - semantic_assessment uses semantic summaries only
# - coarse and focused summaries are interpreted together
# - specific unrelated content becomes INCOMPATIBLE
# - vague content becomes LIMITED, not COMPATIBLE
# - temporal and participation evidence cannot influence the
#   semantic_assessment
# ============================================================

IMPROVED_SEMANTIC_POLICY_BODY = r"""
The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and synchronized segment, you receive:

COARSE SUMMARY:

- speech_content_summary
- apparent_topic

FOCUSED SUMMARY:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The coarse and focused fields are two independently generated semantic
descriptions of the SAME participant segment.

Interpret the coarse and focused fields together.

Do not treat differences between one participant's own coarse and
focused summaries as evidence of semantic incompatibility.

The summaries were independently generated and may sometimes be broad,
imperfect, repetitive, incomplete, or uncertain.

The semantic assessment must be based ONLY on the supplied semantic
summaries.

When assigning semantic_assessment:

- do not use the participation evidence,
- do not use the local temporal evidence,
- do not use the global temporal evidence,
- do not use whether the temporal pattern appears NORMAL or ANOMALOUS,
- and do not allow another reasoning branch to soften or strengthen
  the semantic assessment.

The semantic assessment must independently answer:

Can the two participants' supplied content plausibly belong to the same
real 120-second dyadic conversation?

Internally evaluate:

1. The semantic relationship between Participant A and Participant B
   in Segment 0.
2. The semantic relationship between Participant A and Participant B
   in Segment 1.
3. The complete cross-segment semantic relationship across the full
   120 seconds.

Do not expose these three internal checks as additional output fields.

Do not use a simple vote between Segment 0 and Segment 1.

Evaluate the complete 120-second semantic pattern.

============================================================
SEMANTIC COMPATIBILITY
============================================================

Semantic evidence is COMPATIBLE when the two participants' content can
plausibly belong to the same conversation.

A compatible relationship may include:

- a shared concrete subject,
- compatible people, events, places, experiences, or arguments,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant responding to or reacting to the other,
- one participant supplying context for the other,
- one participant elaborating on a detail introduced by the other,
- different aspects of one shared subject,
- or a coherent topic transition between Segment 0 and Segment 1.

The participants do not need to use identical words.

Exact topic-label matching is not required.

The participants may describe different parts of the same event or
story.

One participant may provide information that is not repeated by the
other participant, provided that the information still fits a plausible
shared conversational context.

A real conversation may naturally change topic during the 120-second
interval.

A topic change does not by itself establish semantic incompatibility.

Assign:

semantic_assessment = COMPATIBLE

only when the concrete content provides credible evidence that the two
participants can plausibly belong to the same conversation.

============================================================
SEMANTIC INCOMPATIBILITY
============================================================

Semantic evidence is INCOMPATIBLE when the participant summaries contain
specific, concrete content that does not support one plausible shared
conversation.

Strong semantic incompatibility may include:

- the participants repeatedly discussing unrelated concrete subjects,
- incompatible people, events, places, activities, products, stories,
  situations, or arguments,
- one participant describing a concrete subject while the other
  describes a clearly unrelated concrete subject,
- both synchronized segments showing semantic mismatch,
- or one synchronized segment showing a strong concrete mismatch while
  the other segment provides no credible evidence of compatibility.

Specific but unrelated content is INCOMPATIBLE.

It must not be treated as LIMITED merely because the model cannot find
a connection.

Do not invent an indirect relationship merely to make the participants
compatible.

Do not assume that two unrelated stories belong to the same conversation
only because both are personal stories.

Do not assume compatibility only because both participants discuss:

- personal experiences,
- personal preferences,
- opinions,
- daily life,
- family,
- relationships,
- products,
- health,
- lifestyle,
- personal well-being,
- or other broad categories.

The fact that two summaries fall under the same broad category is not
sufficient evidence that they describe the same conversation.

Assign:

semantic_assessment = INCOMPATIBLE

when the concrete semantic evidence strongly indicates that the two
participants do not belong to one plausible shared conversation.

============================================================
GENERIC, WEAK, OR UNCERTAIN SEMANTIC EVIDENCE
============================================================

Broad or generic labels such as:

- personal experiences
- personal preferences
- general discussion
- daily life
- opinions
- lifestyle
- personal well-being
- family matters
- emotional topics

must not be treated as evidence of compatibility unless the concrete
summary content also provides a plausible semantic connection.

If one synchronized segment is vague, generic, unclear, incomplete, or
low-confidence, treat that segment internally as insufficient evidence.

An insufficient segment must not automatically support COMPATIBLE.

However, one insufficient segment does not automatically require the
complete semantic assessment to be LIMITED.

Use the other synchronized segment and the complete 120-second pattern:

- If the available concrete evidence clearly supports one plausible
  shared conversation, assign COMPATIBLE.
- If the available concrete evidence shows a strong mismatch and the
  weak segment provides no credible compatibility, assign INCOMPATIBLE.
- If the available summaries are too vague, weak, contradictory, or
  uncertain to establish either compatibility or incompatibility,
  assign LIMITED.

Assign:

semantic_assessment = LIMITED

only when the semantic evidence is genuinely insufficient to determine
whether the two participants belong to the same conversation.

Do not use LIMITED for specific but unrelated content.

============================================================
SEMANTIC ASSESSMENT RULE
============================================================

Use exactly these meanings:

- COMPATIBLE:
  The concrete content supports a plausible shared conversation.

- INCOMPATIBLE:
  The concrete content provides strong evidence of unrelated participant
  records that do not plausibly form one conversation.

- LIMITED:
  The summaries are too generic, unclear, incomplete, contradictory, or
  uncertain to support either conclusion reliably.

The semantic assessment is an independent diagnostic assessment.

Determine it from the semantic summaries before applying the final
combined decision policy.

Semantic COMPATIBLE must not cancel reliable temporal failure.

Semantic INCOMPATIBLE must not be cancelled by apparently normal temporal
coordination.

Semantic LIMITED means that the semantic branch is inconclusive; it does
not itself establish either NORMAL or ANOMALOUS.
""".strip()


# ============================================================
# CONSTRUCT THE COMPLETE PROMPT
#
# Everything before and after SEMANTIC EVIDENCE remains exactly
# as it was in the original full Structured R1 baseline.
# ============================================================

R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE = (
    PROMPT_BEFORE_SEMANTIC
    + SEMANTIC_SECTION_HEADER
    + "\n\n"
    + IMPROVED_SEMANTIC_POLICY_BODY
    + "\n\n"
    + FINAL_DECISION_SECTION_HEADER
    + PROMPT_AFTER_FINAL_DECISION_HEADER
).strip()


# ============================================================
# AUDIT: VERIFY THAT ONLY THE SEMANTIC SECTION CHANGED
# ============================================================

assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    != ORIGINAL_FULL_R1_PROMPT_TEMPLATE
)


# Prefix before semantic section must be exactly identical.
assert (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE[
        :len(PROMPT_BEFORE_SEMANTIC)
    ]
    == PROMPT_BEFORE_SEMANTIC
)


# Everything following the FINAL COMBINED DECISION header must
# remain exactly identical.
improved_after_final = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


original_after_final = (
    ORIGINAL_FULL_R1_PROMPT_TEMPLATE.split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[1]
)


assert improved_after_final == original_after_final


# Required model-facing placeholders must remain present.
REQUIRED_MANUAL_PROMPT_PLACEHOLDERS = [
    "{duration_seconds}",
    "{participant_A_speaks}",
    "{participant_B_speaks}",
    "{local_temporal_features}",
    "{global_shift_features}",
    "{semantic_summaries}",
]


for placeholder in REQUIRED_MANUAL_PROMPT_PLACEHOLDERS:
    assert (
        placeholder
        in R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ), (
        f"Missing required placeholder: {placeholder}"
    )


# No filtered turns or overlap may reappear.
prompt_lower = (
    R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.lower()
)


FORBIDDEN_MANUAL_PROMPT_MARKERS = [
    "filtered turns:",
    "{participant_a_turns}",
    "{participant_b_turns}",
    "turn format and backchannel filtering",
    "clean_overlap_seconds",
    "clean_overlap_percent",
    "filtered overlap",
]


for marker in FORBIDDEN_MANUAL_PROMPT_MARKERS:
    assert marker not in prompt_lower, (
        f"Forbidden content found: {marker}"
    )


# Verify the critical semantic-policy concepts.
REQUIRED_SEMANTIC_POLICY_MARKERS = [
    "specific but unrelated content is INCOMPATIBLE",
    "semantic_assessment = COMPATIBLE",
    "semantic_assessment = INCOMPATIBLE",
    "semantic_assessment = LIMITED",
    "same conversation",
    "plausible",
    "concrete",
    "generic",
    "insufficient",
]


for marker in REQUIRED_SEMANTIC_POLICY_MARKERS:
    assert (
        marker.lower()
        in prompt_lower
    ), (
        f"Missing semantic-policy marker: {marker}"
    )


# ============================================================
# DISPLAY THE EXACT CHANGE
# ============================================================

semantic_policy_diff = "\n".join(
    difflib.unified_diff(
        ORIGINAL_SEMANTIC_POLICY_BODY.splitlines(),
        IMPROVED_SEMANTIC_POLICY_BODY.splitlines(),
        fromfile="original_R1_semantic_policy",
        tofile="improved_R1_semantic_policy",
        lineterm="",
    )
)


print("=" * 100)
print("R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED")
print("=" * 100)

print(
    "Original complete prompt characters:",
    len(
        ORIGINAL_FULL_R1_PROMPT_TEMPLATE
    ),
)

print(
    "Improved complete prompt characters:",
    len(
        R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    ),
)

print(
    "Original semantic-policy characters:",
    len(
        ORIGINAL_SEMANTIC_POLICY_BODY
    ),
)

print(
    "Improved semantic-policy characters:",
    len(
        IMPROVED_SEMANTIC_POLICY_BODY
    ),
)

print(
    "All non-semantic prompt sections preserved exactly:",
    True,
)

print(
    "Filtered turns present:",
    "filtered turns:" in prompt_lower,
)

print(
    "Overlap present:",
    "filtered overlap" in prompt_lower,
)

print("\n" + "=" * 100)
print("SEMANTIC POLICY DIFF")
print("=" * 100)

print(
    semantic_policy_diff
)

R1 IMPROVED SEMANTIC MANUAL PROMPT DEFINED
Original complete prompt characters: 18363
Improved complete prompt characters: 24003
Original semantic-policy characters: 1872
Improved semantic-policy characters: 7512
All non-semantic prompt sections preserved exactly: True
Filtered turns present: False
Overlap present: False

SEMANTIC POLICY DIFF
--- original_R1_semantic_policy
+++ improved_R1_semantic_policy
@@ -3,14 +3,14 @@
 - Segment 0: 0 to 60 seconds
 - Segment 1: 60 to 120 seconds
 
-For each participant and segment, you receive:
-
-Coarse semantic information:
+For each participant and synchronized segment, you receive:
+
+COARSE SUMMARY:
 
 - speech_content_summary
 - apparent_topic
 
-Focused semantic information:
+FOCUSED SUMMARY:
 
 - detailed_speech_summary
 - main_topic
@@ -20,48 +20,217 @@
 - unclear_content
 - confidence
 
-The semantic summaries were independently generated and may be broad,
-imperfect, repetitive, or uncertain.
-
-Evaluate semantic compatibility rather th

## 9. Load the semantic-payload ablation framework


In [ ]:
# ============================================================
# SEMANTIC PAYLOAD ABLATION FRAMEWORK
#
# The full S-CF prompt defined above is the frozen policy baseline.
# Every ablation changes only:
#   1. the semantic fields placed in {semantic_summaries};
#   2. the prompt's inventory of the semantic fields actually supplied.
#
# Participation, temporal evidence, frozen references, semantic
# compatibility criteria, final-decision policy, and output schema
# remain unchanged.
# ============================================================

from collections import Counter
from pathlib import Path

import copy
import difflib
import json
import os
import re
import time


SEMANTIC_ABLATION_ROOT = (
    OUT_DIR
    / "structured_r1_improved_semantic_payload_ablation"
)

SEMANTIC_ABLATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


SEMANTIC_ABLATION_ALL_COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]

SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEMANTIC_ABLATION_SPECS = {
    "S-CF": {
        "title": "Coarse + Focused baseline",
        "round": "group_level",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": list(
            SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
        ),
        "removed_fields": [],
    },

    "S-F": {
        "title": "Focused summaries only",
        "round": "group_level",
        "coarse_fields": [],
        "focused_fields": list(
            SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
        ),
        "removed_fields": [
            "coarse_summary/speech_content_summary",
            "coarse_summary/apparent_topic",
        ],
    },

    "S-C": {
        "title": "Coarse summaries only",
        "round": "group_level",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [],
        "removed_fields": [
            "focused_summary/detailed_speech_summary",
            "focused_summary/main_topic",
            "focused_summary/secondary_topics",
            "focused_summary/key_semantic_details",
            "focused_summary/summary_specificity",
            "focused_summary/unclear_content",
            "focused_summary/confidence",
        ],
    },

    "F1": {
        "title": "Without detailed_speech_summary",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field != "detailed_speech_summary"
        ],
        "removed_fields": [
            "focused_summary/detailed_speech_summary",
        ],
    },

    "F2": {
        "title": "Without main_topic and secondary_topics",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field not in {
                "main_topic",
                "secondary_topics",
            }
        ],
        "removed_fields": [
            "focused_summary/main_topic",
            "focused_summary/secondary_topics",
        ],
    },

    "F3": {
        "title": "Without key_semantic_details",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field != "key_semantic_details"
        ],
        "removed_fields": [
            "focused_summary/key_semantic_details",
        ],
    },

    "F4": {
        "title": "Without reliability metadata",
        "round": "focused_leave_one_group_out",
        "coarse_fields": list(
            SEMANTIC_ABLATION_ALL_COARSE_FIELDS
        ),
        "focused_fields": [
            field
            for field in SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
            if field not in {
                "summary_specificity",
                "unclear_content",
                "confidence",
            }
        ],
        "removed_fields": [
            "focused_summary/summary_specificity",
            "focused_summary/unclear_content",
            "focused_summary/confidence",
        ],
    },
}


SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS = set()
SEMANTIC_ABLATION_CONFIGS = {}
SEMANTIC_ABLATION_EVALUATIONS = {}


# ============================================================
# VALIDATE THE ABLATION SPECIFICATIONS
# ============================================================

assert set(
    SEMANTIC_ABLATION_SPECS
) == {
    "S-CF",
    "S-F",
    "S-C",
    "F1",
    "F2",
    "F3",
    "F4",
}


for ablation_id, spec in (
    SEMANTIC_ABLATION_SPECS.items()
):
    assert (
        spec["coarse_fields"]
        or
        spec["focused_fields"]
    ), (
        f"{ablation_id} removes all semantic evidence."
    )

    assert set(
        spec["coarse_fields"]
    ).issubset(
        SEMANTIC_ABLATION_ALL_COARSE_FIELDS
    )

    assert set(
        spec["focused_fields"]
    ).issubset(
        SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS
    )


# ============================================================
# PROJECT THE FULL SEMANTIC PAYLOAD TO ONE ABLATION
# ============================================================

def project_semantic_summaries_for_ablation(
    semantic_summaries,
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    projected = {}

    for role in [
        "participant_A",
        "participant_B",
    ]:
        role_input = semantic_summaries[
            role
        ]

        role_output = {}

        for segment_name, segment_input in (
            role_input.items()
        ):
            segment_output = {}

            if spec["coarse_fields"]:
                segment_output[
                    "coarse_summary"
                ] = select_exact_fields(
                    segment_input[
                        "coarse_summary"
                    ],
                    spec["coarse_fields"],
                    (
                        f"{ablation_id}/{role}/"
                        f"{segment_name}/coarse_summary"
                    ),
                )

            if spec["focused_fields"]:
                segment_output[
                    "focused_summary"
                ] = select_exact_fields(
                    segment_input[
                        "focused_summary"
                    ],
                    spec["focused_fields"],
                    (
                        f"{ablation_id}/{role}/"
                        f"{segment_name}/focused_summary"
                    ),
                )

            assert segment_output

            role_output[
                segment_name
            ] = segment_output

        projected[
            role
        ] = role_output

    return projected


# ============================================================
# BUILD THE MODEL-FACING PAYLOAD
# ============================================================

def build_semantic_ablation_model_input(
    case,
    ablation_id,
):
    payload = copy.deepcopy(
        build_improved_semantic_model_input(
            case
        )
    )

    payload[
        "semantic_summaries"
    ] = project_semantic_summaries_for_ablation(
        payload[
            "semantic_summaries"
        ],
        ablation_id,
    )

    # The non-semantic representation must remain frozen.
    for role in [
        "participant_A",
        "participant_B",
    ]:
        assert set(
            payload[role]
        ) == {
            "speaks"
        }

    assert set(
        payload[
            "local_temporal_features"
        ]
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS_IMPROVED
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS_IMPROVED
    )

    return payload


# ============================================================
# BUILD AN EVIDENCE INVENTORY MATCHING THE ACTUAL PAYLOAD
# ============================================================

def build_semantic_evidence_inventory_text(
    spec,
):
    lines = []

    if spec["coarse_fields"]:
        lines.extend([
            "COARSE SUMMARY:",
            "",
            *[
                f"- {field}"
                for field in spec[
                    "coarse_fields"
                ]
            ],
        ])

    if (
        spec["coarse_fields"]
        and
        spec["focused_fields"]
    ):
        lines.append("")

    if spec["focused_fields"]:
        lines.extend([
            "FOCUSED SUMMARY:",
            "",
            *[
                f"- {field}"
                for field in spec[
                    "focused_fields"
                ]
            ],
        ])

    lines.append("")

    if (
        spec["coarse_fields"]
        and
        spec["focused_fields"]
    ):
        lines.extend([
            (
                "The supplied coarse and focused fields are semantic "
                "descriptions of the SAME participant segment."
            ),
            "",
            (
                "Interpret all supplied coarse and focused fields together."
            ),
            "",
            (
                "Do not treat differences between one participant's own "
                "coarse and focused summaries as evidence of semantic "
                "incompatibility."
            ),
        ])

    elif spec["focused_fields"]:
        lines.extend([
            (
                "Only the supplied focused semantic fields are available "
                "for each participant segment."
            ),
            "",
            (
                "Interpret the supplied focused fields together as one "
                "semantic description of that participant segment."
            ),
        ])

    else:
        lines.extend([
            (
                "Only the supplied coarse semantic fields are available "
                "for each participant segment."
            ),
            "",
            (
                "Interpret the supplied coarse fields together as one "
                "semantic description of that participant segment."
            ),
        ])

    lines.extend([
        "",
        (
            "The summaries were independently generated and may sometimes "
            "be broad, imperfect, repetitive, incomplete, or uncertain."
        ),
    ])

    return "\n".join(lines).strip()


# ============================================================
# KEEP THE IMPROVED SEMANTIC DECISION POLICY FROZEN
# ============================================================

SEMANTIC_POLICY_CORE_MARKER = (
    "The semantic assessment must be based ONLY on the supplied semantic\n"
    "summaries."
)

assert (
    IMPROVED_SEMANTIC_POLICY_BODY.count(
        SEMANTIC_POLICY_CORE_MARKER
    )
    == 1
)

SEMANTIC_POLICY_CORE = (
    SEMANTIC_POLICY_CORE_MARKER
    + IMPROVED_SEMANTIC_POLICY_BODY.split(
        SEMANTIC_POLICY_CORE_MARKER,
        1,
    )[1]
)


# ============================================================
# UPDATE ONLY THE TOP-LEVEL AVAILABLE-EVIDENCE INVENTORY
# ============================================================

def replace_available_evidence_inventory(
    prompt_prefix,
    spec,
):
    start_marker = "You receive:\n\n"
    end_marker = "\n\nUse only the supplied evidence."

    assert prompt_prefix.count(
        start_marker
    ) == 1

    assert prompt_prefix.count(
        end_marker
    ) == 1

    before, remainder = prompt_prefix.split(
        start_marker,
        1,
    )

    _, after = remainder.split(
        end_marker,
        1,
    )

    evidence_items = [
        "Whether each participant speaks during the 120-second interval.",
        "Local turn-handoff offset-distribution features.",
        "Global temporal alignment-shift features.",
    ]

    if spec["coarse_fields"]:
        evidence_items.append(
            "Coarse semantic summaries for two synchronized 60-second segments."
        )

    if spec["focused_fields"]:
        evidence_items.append(
            "Focused semantic summaries for the same two segments."
        )

    evidence_items.append(
        (
            "Frozen temporal reference statistics calculated only from\n"
            "   separate NORMAL dyadic conversations."
        )
    )

    numbered = "\n".join(
        f"{index}. {item}"
        for index, item in enumerate(
            evidence_items,
            start=1,
        )
    )

    return (
        before
        + start_marker
        + numbered
        + end_marker
        + after
    )


# ============================================================
# BUILD THE COMPLETE PROMPT TEMPLATE FOR ONE ABLATION
# ============================================================

def build_semantic_ablation_prompt_template(
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    # The S-CF control must remain exactly identical to the
    # already executed improved-semantic baseline.
    if ablation_id == "S-CF":
        return (
            R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
        )

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    updated_prefix = (
        replace_available_evidence_inventory(
            PROMPT_BEFORE_SEMANTIC,
            spec,
        )
    )

    semantic_inventory = (
        build_semantic_evidence_inventory_text(
            spec
        )
    )

    semantic_body = (
        "The 120-second interval is divided into:\n\n"
        "- Segment 0: 0 to 60 seconds\n"
        "- Segment 1: 60 to 120 seconds\n\n"
        "For each participant and synchronized segment, you receive:\n\n"
        + semantic_inventory
        + "\n\n"
        + SEMANTIC_POLICY_CORE
    )

    prompt_template = (
        updated_prefix
        + SEMANTIC_SECTION_HEADER
        + "\n\n"
        + semantic_body
        + "\n\n"
        + FINAL_DECISION_SECTION_HEADER
        + PROMPT_AFTER_FINAL_DECISION_HEADER
    ).strip()

    return prompt_template


# ============================================================
# VALIDATE THAT EACH PROMPT MATCHES ITS SEMANTIC PAYLOAD
# ============================================================

def validate_semantic_ablation_prompt(
    ablation_id,
    prompt_template,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    template = validate_manual_improved_semantic_prompt(
        prompt_template
    )

    spec = SEMANTIC_ABLATION_SPECS[
        ablation_id
    ]

    semantic_section = template.split(
        SEMANTIC_SECTION_HEADER,
        1,
    )[1].split(
        FINAL_DECISION_SECTION_HEADER,
        1,
    )[0]

    # Every supplied field must be documented in the semantic section.
    for field in (
        spec["coarse_fields"]
        + spec["focused_fields"]
    ):
        assert field in semantic_section, (
            f"{ablation_id}: supplied field missing from prompt: {field}"
        )

    # Every removed field must be absent from the evidence inventory.
    inventory = semantic_section.split(
        SEMANTIC_POLICY_CORE_MARKER,
        1,
    )[0]

    for field in (
        set(SEMANTIC_ABLATION_ALL_COARSE_FIELDS)
        - set(spec["coarse_fields"])
    ):
        assert field not in inventory, (
            f"{ablation_id}: removed coarse field remains in inventory: {field}"
        )

    for field in (
        set(SEMANTIC_ABLATION_ALL_FOCUSED_FIELDS)
        - set(spec["focused_fields"])
    ):
        assert field not in inventory, (
            f"{ablation_id}: removed focused field remains in inventory: {field}"
        )

    # The policy after the evidence inventory must remain frozen.
    assert (
        semantic_section.split(
            SEMANTIC_POLICY_CORE_MARKER,
            1,
        )[1].strip()
        ==
        IMPROVED_SEMANTIC_POLICY_BODY.split(
            SEMANTIC_POLICY_CORE_MARKER,
            1,
        )[1].strip()
    )

    prompt_lower = template.lower()

    for forbidden_marker in [
        "filtered turns:",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert forbidden_marker not in prompt_lower

    return template


# ============================================================
# RENDER ONE MODEL-FACING PROMPT
# ============================================================

def build_semantic_ablation_prompt(
    case,
    config,
):
    payload = build_semantic_ablation_model_input(
        case,
        config[
            "semantic_ablation_id"
        ],
    )

    prompt = render_manual_improved_semantic_prompt(
        config[
            "reasoning_prompt_template"
        ],
        payload,
    )

    return prompt, payload


# ============================================================
# PREPARE ONE ABLATION CONFIGURATION
# ============================================================

def prepare_semantic_ablation_experiment(
    ablation_id,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    spec = copy.deepcopy(
        SEMANTIC_ABLATION_SPECS[
            ablation_id
        ]
    )

    prompt_template = (
        build_semantic_ablation_prompt_template(
            ablation_id
        )
    )

    prompt_template = (
        validate_semantic_ablation_prompt(
            ablation_id,
            prompt_template,
        )
    )

    slug = (
        ablation_id
        .lower()
        .replace("-", "_")
    )

    experiment_version = (
        "structured_r1_improved_semantic_payload_ablation_"
        + slug
    )

    experiment_dir = (
        SEMANTIC_ABLATION_ROOT
        / slug
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": experiment_dir / "predictions_cache.json",
        "predictions_csv": experiment_dir / "predictions_all_400.csv",
        "metrics_json": experiment_dir / "metrics.json",
        "confusion_matrix_csv": experiment_dir / "confusion_matrix.csv",
        "family_metrics_csv": experiment_dir / "metrics_by_case_family.csv",
        "variant_metrics_csv": experiment_dir / "metrics_by_case_variant.csv",
        "errors_csv": experiment_dir / "classification_errors.csv",
        "reasoning_assessments_csv": experiment_dir / "reasoning_assessments.csv",
        "reasoning_inconsistencies_csv": experiment_dir / "reasoning_inconsistencies.csv",
        "assessment_distributions_csv": experiment_dir / "assessment_distributions.csv",
        "prompt_template": experiment_dir / "prompt_template.txt",
        "prompt_diff": experiment_dir / "prompt_diff_vs_S_CF.txt",
        "rendered_prompt_preview": experiment_dir / "rendered_prompt_preview.txt",
        "experiment_manifest": experiment_dir / "experiment_manifest.json",
    }

    reasoning_prompt_sha256 = sha256_text(
        prompt_template
    )

    baseline_prompt_sha256 = sha256_text(
        R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE
    )

    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )

    prompt_diff = "\n".join(
        difflib.unified_diff(
            R1_IMPROVED_SEMANTIC_MANUAL_PROMPT_TEMPLATE.splitlines(),
            prompt_template.splitlines(),
            fromfile="S-CF_full_semantic_payload",
            tofile=f"{ablation_id}_{spec['title']}",
            lineterm="",
        )
    )

    paths[
        "prompt_template"
    ].write_text(
        prompt_template,
        encoding="utf-8",
    )

    paths[
        "prompt_diff"
    ].write_text(
        prompt_diff,
        encoding="utf-8",
    )

    manifest = {
        "experiment_version": experiment_version,
        "experiment_title": (
            "Structured R1 — Improved Semantic Policy — "
            f"Semantic Payload Ablation {ablation_id}: {spec['title']}"
        ),
        "ablation_id": (
            f"R1_IMPROVED_SEMANTIC_PAYLOAD_{ablation_id}"
        ),
        "semantic_ablation_id": ablation_id,
        "semantic_ablation_round": spec[
            "round"
        ],
        "semantic_input": ablation_id,
        "semantic_payload_title": spec[
            "title"
        ],
        "semantic_coarse_fields": list(
            spec["coarse_fields"]
        ),
        "semantic_focused_fields": list(
            spec["focused_fields"]
        ),
        "semantic_removed_fields": list(
            spec["removed_fields"]
        ),
        "focused_summaries_used": bool(
            spec["focused_fields"]
        ),
        "source_experiment": (
            IMPROVED_SEMANTIC_EXPERIMENT_VERSION
        ),
        "model_id": MODEL_ID,
        "max_new_tokens": MAX_NEW_TOKENS_REASONING,
        "reasoning_prompt_sha256": reasoning_prompt_sha256,
        "baseline_prompt_sha256": baseline_prompt_sha256,
        "normal_reference_sha256": normal_reference_sha256,
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "participant_model_input": "speaks_only",
        "local_model_input": "complete_offset_distribution_only",
        "global_model_input": "all_five_global_features",
        "temporal_profiles_used": "frozen_NORMAL_only",
        "assessment_policy": (
            "structured_R1_with_frozen_improved_semantic_policy"
        ),
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "prompt_definition_method": (
            "frozen_policy_with_payload_specific_manual_inspection"
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": experiment_dir,
        "reasoning_prompt_template": prompt_template,
        "paths": paths,
    }

    SEMANTIC_ABLATION_CONFIGS[
        ablation_id
    ] = config

    print("=" * 100)
    print(
        f"SEMANTIC ABLATION {ablation_id} — CONFIGURATION READY"
    )
    print("=" * 100)
    print("Title:", spec["title"])
    print("Coarse fields:", spec["coarse_fields"])
    print("Focused fields:", spec["focused_fields"])
    print("Removed fields:", spec["removed_fields"])
    print("Prompt SHA256:", reasoning_prompt_sha256)
    print("Output directory:", experiment_dir)
    print(
        "Existing cache:",
        paths["prediction_cache"].exists(),
    )

    return config


# ============================================================
# REQUIRED EXACT PROMPT INSPECTION
# ============================================================

def inspect_semantic_ablation_prompt(
    config,
    *,
    case_index=0,
    prefer_wrong_partner=True,
):
    ablation_id = config[
        "semantic_ablation_id"
    ]

    if prefer_wrong_partner:
        candidate_cases = sorted(
            [
                case
                for case in consolidation_cases
                if get_case_family(case)
                == "wrong_partner"
            ],
            key=lambda case: str(
                case["case_id"]
            ),
        )
    else:
        candidate_cases = sorted(
            consolidation_cases,
            key=lambda case: str(
                case["case_id"]
            ),
        )

    assert (
        0
        <= case_index
        < len(candidate_cases)
    )

    case = candidate_cases[
        case_index
    ]

    prompt, payload = (
        build_semantic_ablation_prompt(
            case,
            config,
        )
    )

    config[
        "paths"
    ][
        "rendered_prompt_preview"
    ].write_text(
        prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        f"SEMANTIC ABLATION {ablation_id} — EXACT PROMPT INSPECTION"
    )
    print("=" * 100)
    print("Inspection case ID:", case["case_id"])
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print("Participant evidence:", "speaks only")
    print("Local evidence:", "complete offset distribution")
    print("Global evidence:", "all five global features")
    print(
        "Coarse fields supplied:",
        config["semantic_coarse_fields"],
    )
    print(
        "Focused fields supplied:",
        config["semantic_focused_fields"],
    )
    print(
        "Removed semantic fields:",
        config["semantic_removed_fields"],
    )
    print("Filtered turns supplied:", False)
    print("Overlap supplied:", False)

    print("\n" + "=" * 100)
    print("EXACT MODEL-FACING PAYLOAD")
    print("=" * 100)
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT RENDERED MODEL PROMPT")
    print("=" * 100)
    print(prompt)

    SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS.add(
        config["experiment_version"]
    )

    print(
        "\nInspection unlocked:",
        config["experiment_version"]
        in SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS,
    )
    print(
        "Saved rendered prompt:",
        config[
            "paths"
        ][
            "rendered_prompt_preview"
        ],
    )

    return {
        "case": case,
        "payload": payload,
        "prompt": prompt,
    }


# ============================================================
# CACHE
# ============================================================

def create_semantic_ablation_cache(
    config,
):
    return {
        "experiment_version": config[
            "experiment_version"
        ],
        "experiment_title": config[
            "experiment_title"
        ],
        "ablation_id": config[
            "ablation_id"
        ],
        "semantic_ablation_id": config[
            "semantic_ablation_id"
        ],
        "source_experiment": config[
            "source_experiment"
        ],
        "model_id": config[
            "model_id"
        ],
        "reasoning_prompt_sha256": config[
            "reasoning_prompt_sha256"
        ],
        "baseline_prompt_sha256": config[
            "baseline_prompt_sha256"
        ],
        "normal_reference_sha256": config[
            "normal_reference_sha256"
        ],
        "semantic_input": config[
            "semantic_input"
        ],
        "semantic_coarse_fields": list(
            config["semantic_coarse_fields"]
        ),
        "semantic_focused_fields": list(
            config["semantic_focused_fields"]
        ),
        "semantic_removed_fields": list(
            config["semantic_removed_fields"]
        ),
        "focused_summaries_used": bool(
            config["focused_summaries_used"]
        ),
        "filtered_turns_supplied": False,
        "overlap_supplied": False,
        "temporal_profiles_used": config[
            "temporal_profiles_used"
        ],
        "assessment_policy": config[
            "assessment_policy"
        ],
        "schema_keys": list(
            REASONING_SCHEMA_KEYS
        ),
        "max_new_tokens": MAX_NEW_TOKENS_REASONING,
        "created_at_utc": reasoning_utc_now(),
        "updated_at_utc": reasoning_utc_now(),
        "records": {},
    }


# ============================================================
# RUN ONE EXPERIMENT — MANUAL CALL ONLY
# ============================================================

def run_semantic_ablation_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config["experiment_version"]
        in SEMANTIC_ABLATION_INSPECTED_EXPERIMENTS
    ), (
        "Run the exact prompt-inspection cell for this ablation before inference."
    )

    prediction_cache_path = config[
        "paths"
    ][
        "prediction_cache"
    ]

    expected_cache = create_semantic_ablation_cache(
        config
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "semantic_ablation_id",
            "source_experiment",
            "model_id",
            "reasoning_prompt_sha256",
            "baseline_prompt_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "semantic_coarse_fields",
            "semantic_focused_fields",
            "semantic_removed_fields",
            "focused_summaries_used",
            "filtered_turns_supplied",
            "overlap_supplied",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[key]
                == expected_cache[key]
            ), (
                f"Cache mismatch for {key}. "
                "Use the experiment's own output directory or remove the incompatible cache."
            )

        print("Resuming cache:", prediction_cache_path)
        print(
            "Existing records:",
            len(prediction_cache["records"]),
        )

    else:
        prediction_cache = expected_cache

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print("Created cache:", prediction_cache_path)

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case["case_id"]
        ),
    )

    assert len(ordered_cases) == 400

    for case in tqdm(
        ordered_cases,
        desc=config["experiment_version"],
    ):
        case_id = str(
            case["case_id"]
        )

        prompt, payload = (
            build_semantic_ablation_prompt(
                case,
                config,
            )
        )

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        prompt_sha256 = sha256_text(
            prompt
        )

        payload_sha256 = sha256_text(
            canonical_json(
                payload
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(case_id)
        )

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )

            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == payload_sha256
            )

            continue

        started = time.perf_counter()

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case["source_group_id"]
            ),
            "case_family": get_case_family(
                case
            ),
            "case_variant": str(
                case["case_variant"]
            ),
            "gold_binary_label": str(
                case["gold_binary_label"]
            ).upper(),
            "semantic_ablation_id": config[
                "semantic_ablation_id"
            ],
            "semantic_coarse_fields": list(
                config["semantic_coarse_fields"]
            ),
            "semantic_focused_fields": list(
                config["semantic_focused_fields"]
            ),
            "prompt_sha256": prompt_sha256,
            "input_payload_sha256": payload_sha256,
            "semantic_input": config[
                "semantic_input"
            ],
            "focused_summaries_used": bool(
                config["focused_summaries_used"]
            ),
            "input_token_count": input_token_count,
            "max_new_tokens": MAX_NEW_TOKENS_REASONING,
            "raw_output": raw_output,
            "prediction": parsed_result[
                "prediction"
            ],
            "participation_assessment": parsed_result[
                "participation_assessment"
            ],
            "local_temporal_assessment": parsed_result[
                "local_temporal_assessment"
            ],
            "global_temporal_assessment": parsed_result[
                "global_temporal_assessment"
            ],
            "temporal_assessment": parsed_result[
                "temporal_assessment"
            ],
            "semantic_assessment": parsed_result[
                "semantic_assessment"
            ],
            "decisive_dimension": parsed_result[
                "decisive_dimension"
            ],
            "parse_mode": parsed_result[
                "parse_mode"
            ],
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": parsed_result[
                "schema_errors"
            ],
            "parsed_output": parsed_result[
                "parsed_output"
            ],
            "generation_error": generation_error,
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": reasoning_utc_now(),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 100)
    print(
        f"SEMANTIC ABLATION {config['semantic_ablation_id']} — INFERENCE COMPLETE"
    )
    print("=" * 100)
    print(
        "Cached records:",
        len(prediction_cache["records"]),
    )
    print("Prediction cache:", prediction_cache_path)

    return prediction_cache


# ============================================================
# REGISTER AN EVALUATION FOR FINAL COMPARISON
# ============================================================

def register_semantic_ablation_evaluation(
    ablation_id,
    evaluation,
):
    assert ablation_id in SEMANTIC_ABLATION_SPECS

    SEMANTIC_ABLATION_EVALUATIONS[
        ablation_id
    ] = evaluation

    print(
        "Registered evaluation:",
        ablation_id,
    )

    return evaluation


print("Semantic ablation framework ready.")
print(
    "Experiments:",
    list(SEMANTIC_ABLATION_SPECS),
)
print(
    "Run each prompt-inspection cell manually before its inference cell."
)


Semantic ablation framework ready.
Experiments: ['S-CF', 'S-F', 'S-C', 'F1', 'F2', 'F3', 'F4']
Run each prompt-inspection cell manually before its inference cell.


## 10. Prepare and inspect the exact frozen F1 prompt

This is the same F1 configuration that removed only
`detailed_speech_summary`.

Read the printed payload and rendered prompt before continuing.


In [ ]:
# ============================================================
# F1 — PREPARE AND PRINT THE EXACT PROMPT
#
# This cell does not run model inference.
# Read the full payload and rendered prompt before continuing.
# ============================================================

F1_CONFIG = prepare_semantic_ablation_experiment(
    "F1"
)

F1_PROMPT_INSPECTION = inspect_semantic_ablation_prompt(
    F1_CONFIG,
    case_index=0,
    prefer_wrong_partner=True,
)


SEMANTIC ABLATION F1 — CONFIGURATION READY
Title: Without detailed_speech_summary
Coarse fields: ['speech_content_summary', 'apparent_topic']
Focused fields: ['main_topic', 'secondary_topics', 'key_semantic_details', 'summary_specificity', 'unclear_content', 'confidence']
Removed fields: ['focused_summary/detailed_speech_summary']
Prompt SHA256: bf50d017f027995c6eaae81adad6b04d263530d25f2f4361e3ee3d9f19791355
Output directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1
Existing cache: True
SEMANTIC ABLATION F1 — EXACT PROMPT INSPECTION
Inspection case ID: consolidation_wrong_partner_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence: speaks only
Local evidence: complete offset distribution
Global evidence: all five global features
Coarse fields supplied: ['speech_content_summary', 'apparent_topic']
Focused fields supplied: ['main_topic', 'secondary_topics', 'key_semantic_d

## 11. Load and audit the frozen F1 prediction cache

The cache must contain the exact 400 deterministic F1 predictions already
generated by the source experiment. It provides:

1. the baseline to compare against;
2. the teacher outputs for preservation loss;
3. the wrong-partner semantic strata used for the grouped split.


In [ ]:

# ============================================================
# LOAD AND AUDIT THE EXACT F1 BASELINE CACHE
# ============================================================

F1_CACHE_PATH = F1_CONFIG["paths"]["prediction_cache"]

assert F1_CACHE_PATH.exists(), (
    "The exact F1 prediction cache is required before fine-tuning.\n"
    f"Expected path: {F1_CACHE_PATH}\n"
    "Run the original F1 experiment first if this file is missing."
)

F1_BASELINE_CACHE = json.loads(
    F1_CACHE_PATH.read_text(encoding="utf-8")
)

assert isinstance(F1_BASELINE_CACHE, dict)
assert isinstance(F1_BASELINE_CACHE.get("records"), dict)
assert len(F1_BASELINE_CACHE["records"]) == 400

F1_RECORDS = {
    str(case_id): record
    for case_id, record in F1_BASELINE_CACHE["records"].items()
}

for case in consolidation_cases:
    case_id = str(case["case_id"])
    assert case_id in F1_RECORDS

    record = F1_RECORDS[case_id]

    assert record.get("schema_exact") is True, (
        f"F1 record is not exact-schema: {case_id}"
    )

    assert record.get("prediction") in LABELS
    assert record.get("semantic_assessment") in {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    }

print("=" * 100)
print("EXACT F1 CACHE LOADED")
print("=" * 100)
print("Path:", F1_CACHE_PATH)
print("Records:", len(F1_RECORDS))
print("Prompt SHA256:", F1_CONFIG["reasoning_prompt_sha256"])

F1_BASELINE_EVALUATION = evaluate_reasoning_experiment(
    F1_CONFIG
)


EXACT F1 CACHE LOADED
Path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
Records: 400
Prompt SHA256: bf50d017f027995c6eaae81adad6b04d263530d25f2f4361e3ee3d9f19791355
Structured R1 — Improved Semantic Policy — Semantic Payload Ablation F1: Without detailed_speech_summary — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8900
Balanced accuracy: 0.8433
ANOMALOUS precision: 0.9183
ANOMALOUS recall: 0.9367
ANOMALOUS F1: 0.9274
NORMAL recall / specificity: 0.7500
MCC: 0.7013
Matched source-group exact rate: 0.6000
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,75,25
Gold ANOMALOUS,19,281



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.797872,0.750000,0.773196,100.00
ANOMALOUS,0.918301,0.936667,0.927393,300.00
accuracy,0.890000,0.890000,0.890000,0.89
macro avg,0.858086,0.843333,0.850294,400.00
weighted avg,0.888194,0.890000,0.888844,400.00



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,10,40,40,0.80,0.80,1.0
1,lag_3sec,50,50,0,5,45,45,0.90,0.90,1.0
2,normal,100,100,0,75,25,75,0.75,0.75,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,4,96,96,0.96,0.96,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,ANOMALOUS,266
3,local_temporal_assessment,NORMAL,132
4,local_temporal_assessment,LIMITED,2
5,global_temporal_assessment,ANOMALOUS,284
6,global_temporal_assessment,NORMAL,114
7,global_temporal_assessment,LIMITED,2
8,temporal_assessment,ANOMALOUS,284
9,temporal_assessment,NORMAL,114



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/prompt_diff_vs_S_CF.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/classification_errors.csv

All 400 cases produced valid bina

## 12. Create the deterministic grouped train/validation/test split

The four wrong-partner cases that F1 predicted as NORMAL are forced into the
held-out set. This avoids creating a training sequence whose new semantic
target is `INCOMPATIBLE` while its teacher-preserved final label is `NORMAL`.

The 40 training wrong-partner cases are selected with the requested F1
baseline composition:

- 16 baseline `INCOMPATIBLE`
- 14 baseline `COMPATIBLE`
- 10 baseline `LIMITED`


In [ ]:

# ============================================================
# DETERMINISTIC GROUPED SPLIT
#
# Every source_group_id contains exactly four variants:
#   NORMAL, LAG, WRONG_PARTNER, SILENT_PARTNER.
#
# All four variants from one source group remain in the same split.
#
# The split is stratified using ONLY the frozen F1 semantic output
# of the WRONG_PARTNER variant. The four wrong-partner cases missed
# by F1 are deliberately held out from train/validation, avoiding
# a contradictory preservation target:
#   semantic target = INCOMPATIBLE but teacher label = NORMAL.
# ============================================================

from collections import Counter, defaultdict
import random
import numpy as np

SPLIT_SEED = 42

TRAIN_GROUP_COUNT = 40
VALIDATION_GROUP_COUNT = 10
TEST_GROUP_COUNT = 50

# Exact requested training composition among the 40 wrong-partner cases.
TRAIN_WP_BASELINE_COUNTS = {
    "INCOMPATIBLE": 16,
    "COMPATIBLE": 14,
    "LIMITED": 10,
}

# A small stratified validation set.
VALIDATION_WP_BASELINE_COUNTS = {
    "INCOMPATIBLE": 4,
    "COMPATIBLE": 3,
    "LIMITED": 3,
}

assert sum(TRAIN_WP_BASELINE_COUNTS.values()) == TRAIN_GROUP_COUNT
assert sum(VALIDATION_WP_BASELINE_COUNTS.values()) == VALIDATION_GROUP_COUNT

cases_by_group = defaultdict(list)

for case in consolidation_cases:
    cases_by_group[str(case["source_group_id"])].append(case)

assert len(cases_by_group) == 100

for group_id, group_cases in cases_by_group.items():
    family_counter = Counter(
        get_case_family(case)
        for case in group_cases
    )

    assert family_counter == {
        "normal": 1,
        "lag": 1,
        "wrong_partner": 1,
        "silent_partner": 1,
    }, (
        f"Unexpected group structure for {group_id}: {family_counter}"
    )

wrong_partner_case_by_group = {}

for group_id, group_cases in cases_by_group.items():
    wp_cases = [
        case
        for case in group_cases
        if get_case_family(case) == "wrong_partner"
    ]

    assert len(wp_cases) == 1
    wrong_partner_case_by_group[group_id] = wp_cases[0]

eligible_groups_by_wp_semantic = defaultdict(list)
held_out_missed_wrong_partner_groups = []

for group_id, wp_case in wrong_partner_case_by_group.items():
    case_id = str(wp_case["case_id"])
    teacher = F1_RECORDS[case_id]

    if teacher["prediction"] != "ANOMALOUS":
        held_out_missed_wrong_partner_groups.append(group_id)
        continue

    semantic_value = teacher["semantic_assessment"]

    assert semantic_value in {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    }

    eligible_groups_by_wp_semantic[semantic_value].append(group_id)

print("=" * 100)
print("F1 WRONG-PARTNER GROUPS AVAILABLE FOR TRAIN/VALIDATION")
print("=" * 100)

for semantic_value in [
    "INCOMPATIBLE",
    "COMPATIBLE",
    "LIMITED",
]:
    print(
        semantic_value,
        len(eligible_groups_by_wp_semantic[semantic_value]),
    )

print(
    "Wrong-partner groups missed by F1 and forced into test:",
    len(held_out_missed_wrong_partner_groups),
)

rng = random.Random(SPLIT_SEED)

for semantic_value in eligible_groups_by_wp_semantic:
    rng.shuffle(
        eligible_groups_by_wp_semantic[semantic_value]
    )

train_group_ids = []
validation_group_ids = []

for semantic_value in [
    "INCOMPATIBLE",
    "COMPATIBLE",
    "LIMITED",
]:
    group_pool = eligible_groups_by_wp_semantic[semantic_value]

    train_n = TRAIN_WP_BASELINE_COUNTS[semantic_value]
    validation_n = VALIDATION_WP_BASELINE_COUNTS[semantic_value]

    assert len(group_pool) >= train_n + validation_n, (
        f"Not enough {semantic_value} wrong-partner groups."
    )

    train_group_ids.extend(
        group_pool[:train_n]
    )

    validation_group_ids.extend(
        group_pool[train_n:train_n + validation_n]
    )

all_group_ids = set(cases_by_group)

train_group_ids = sorted(set(train_group_ids))
validation_group_ids = sorted(set(validation_group_ids))

test_group_ids = sorted(
    all_group_ids
    - set(train_group_ids)
    - set(validation_group_ids)
)

assert len(train_group_ids) == TRAIN_GROUP_COUNT
assert len(validation_group_ids) == VALIDATION_GROUP_COUNT
assert len(test_group_ids) == TEST_GROUP_COUNT

assert not (
    set(train_group_ids)
    & set(validation_group_ids)
)

assert not (
    set(train_group_ids)
    & set(test_group_ids)
)

assert not (
    set(validation_group_ids)
    & set(test_group_ids)
)

assert set(
    held_out_missed_wrong_partner_groups
).issubset(
    set(test_group_ids)
)

def split_cases(group_ids, families=None):
    group_ids = set(group_ids)

    selected = [
        case
        for case in consolidation_cases
        if str(case["source_group_id"]) in group_ids
    ]

    if families is not None:
        families = set(families)

        selected = [
            case
            for case in selected
            if get_case_family(case) in families
        ]

    return sorted(
        selected,
        key=lambda case: str(case["case_id"]),
    )

# Silent cases are excluded from semantic-targeted training/validation.
TRAIN_CASES = split_cases(
    train_group_ids,
    families={
        "normal",
        "lag",
        "wrong_partner",
    },
)

VALIDATION_CASES = split_cases(
    validation_group_ids,
    families={
        "normal",
        "lag",
        "wrong_partner",
    },
)

# Evaluation uses every remaining case, including all silent partners.
TEST_CASES = split_cases(
    test_group_ids,
    families={
        "normal",
        "lag",
        "wrong_partner",
        "silent_partner",
    },
)

assert len(TRAIN_CASES) == 120
assert len(VALIDATION_CASES) == 30
assert len(TEST_CASES) == 200

def family_counts_for_cases(cases):
    return Counter(
        get_case_family(case)
        for case in cases
    )

print("\n" + "=" * 100)
print("GROUPED SPLIT COMPLETE")
print("=" * 100)
print("Train groups:", len(train_group_ids))
print("Validation groups:", len(validation_group_ids))
print("Test groups:", len(test_group_ids))
print("Train cases:", family_counts_for_cases(TRAIN_CASES))
print("Validation cases:", family_counts_for_cases(VALIDATION_CASES))
print("Test cases:", family_counts_for_cases(TEST_CASES))


F1 WRONG-PARTNER GROUPS AVAILABLE FOR TRAIN/VALIDATION
INCOMPATIBLE 39
COMPATIBLE 32
LIMITED 25
Wrong-partner groups missed by F1 and forced into test: 4

GROUPED SPLIT COMPLETE
Train groups: 40
Validation groups: 10
Test groups: 50
Train cases: Counter({'lag': 40, 'normal': 40, 'wrong_partner': 40})
Validation cases: Counter({'lag': 10, 'normal': 10, 'wrong_partner': 10})
Test cases: Counter({'lag': 50, 'normal': 50, 'silent_partner': 50, 'wrong_partner': 50})


In [ ]:
# ============================================================
# AUDIT: FROZEN F1 PREDICTIONS ON THE SELECTED TEST SET
#
# Uses the already-existing F1 prediction cache.
# No new model inference is performed.
# ============================================================

import pandas as pd

f1_test_rows = []

for case in TEST_CASES:
    case_id = str(case["case_id"])
    f1_record = F1_RECORDS[case_id]

    gold_label = str(
        case["gold_binary_label"]
    ).upper()

    prediction = str(
        f1_record["prediction"]
    ).upper()

    f1_test_rows.append({
        "case_id": case_id,
        "source_group_id": str(
            case["source_group_id"]
        ),
        "case_family": get_case_family(case),
        "case_variant": str(
            case["case_variant"]
        ),
        "gold_label": gold_label,
        "f1_prediction": prediction,
        "correct": prediction == gold_label,

        "participation_assessment": (
            f1_record["participation_assessment"]
        ),
        "local_temporal_assessment": (
            f1_record["local_temporal_assessment"]
        ),
        "global_temporal_assessment": (
            f1_record["global_temporal_assessment"]
        ),
        "temporal_assessment": (
            f1_record["temporal_assessment"]
        ),
        "semantic_assessment": (
            f1_record["semantic_assessment"]
        ),
        "decisive_dimension": (
            f1_record["decisive_dimension"]
        ),

        "schema_exact": bool(
            f1_record["schema_exact"]
        ),
    })

frozen_f1_selected_test_df = pd.DataFrame(
    f1_test_rows
).sort_values(
    [
        "case_family",
        "case_id",
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# BASIC ASSERTIONS
# ------------------------------------------------------------

assert len(frozen_f1_selected_test_df) == 200

assert (
    frozen_f1_selected_test_df[
        "case_family"
    ].value_counts().to_dict()
    ==
    {
        "normal": 50,
        "lag": 50,
        "wrong_partner": 50,
        "silent_partner": 50,
    }
)

assert frozen_f1_selected_test_df[
    "schema_exact"
].all()

print("=" * 100)
print("FROZEN F1 — SELECTED TEST SET")
print("=" * 100)

print(
    "\nNumber of test cases:",
    len(frozen_f1_selected_test_df),
)

print(
    "Exact-schema predictions:",
    int(
        frozen_f1_selected_test_df[
            "schema_exact"
        ].sum()
    ),
)

# ------------------------------------------------------------
# 1. FINAL F1 PREDICTIONS PER FAMILY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL PREDICTIONS PER CASE FAMILY")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_selected_test_df[
            "case_family"
        ],
        columns=frozen_f1_selected_test_df[
            "f1_prediction"
        ],
        margins=True,
    )
)

# ------------------------------------------------------------
# 2. ACCURACY PER FAMILY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("F1 ACCURACY ON THE SELECTED TEST SET")
print("=" * 100)

f1_test_family_performance = (
    frozen_f1_selected_test_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=("case_id", "count"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
        predicted_NORMAL=(
            "f1_prediction",
            lambda values: (
                values == "NORMAL"
            ).sum(),
        ),
        predicted_ANOMALOUS=(
            "f1_prediction",
            lambda values: (
                values == "ANOMALOUS"
            ).sum(),
        ),
    )
)

display(f1_test_family_performance)

# ------------------------------------------------------------
# 3. SEMANTIC ASSESSMENTS PER FAMILY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("F1 SEMANTIC ASSESSMENTS ON THE SELECTED TEST SET")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_selected_test_df[
            "case_family"
        ],
        columns=frozen_f1_selected_test_df[
            "semantic_assessment"
        ],
        margins=True,
    )
)

# ------------------------------------------------------------
# 4. TEMPORAL ASSESSMENTS PER FAMILY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("F1 TEMPORAL ASSESSMENTS ON THE SELECTED TEST SET")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_selected_test_df[
            "case_family"
        ],
        columns=frozen_f1_selected_test_df[
            "temporal_assessment"
        ],
        margins=True,
    )
)

# ------------------------------------------------------------
# 5. DECISIVE DIMENSION PER FAMILY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("F1 DECISIVE DIMENSION ON THE SELECTED TEST SET")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_selected_test_df[
            "case_family"
        ],
        columns=frozen_f1_selected_test_df[
            "decisive_dimension"
        ],
        margins=True,
    )
)

# ------------------------------------------------------------
# 6. COMPLETE CASE-LEVEL TABLE
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPLETE CASE-LEVEL F1 TEST PREDICTIONS")
print("=" * 100)

display(
    frozen_f1_selected_test_df
)

FROZEN F1 — SELECTED TEST SET

Number of test cases: 200
Exact-schema predictions: 200

FINAL PREDICTIONS PER CASE FAMILY


f1_prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,43,7,50
normal,12,38,50
silent_partner,50,0,50
wrong_partner,46,4,50
All,151,49,200



F1 ACCURACY ON THE SELECTED TEST SET


,case_family,total_cases,correct_predictions,accuracy,predicted_NORMAL,predicted_ANOMALOUS
0,lag,50,43,0.86,7,43
1,normal,50,38,0.76,38,12
2,silent_partner,50,50,1.00,0,50
3,wrong_partner,50,46,0.92,4,46



F1 SEMANTIC ASSESSMENTS ON THE SELECTED TEST SET


...

temporal_assessment,ANOMALOUS,LIMITED,NORMAL,All
case_family,,,,
lag,42,0,8,50
normal,9,1,40,50
silent_partner,49,1,0,50
wrong_partner,37,0,13,50
All,137,2,61,200



F1 DECISIVE DIMENSION ON THE SELECTED TEST SET


decisive_dimension,PARTICIPATION,SEMANTIC,TEMPORAL,All
case_family,,,,
lag,0,8,42,50
normal,0,41,9,50
silent_partner,28,10,12,50
wrong_partner,0,13,37,50
All,28,72,100,200



COMPLETE CASE-LEVEL F1 TEST PREDICTIONS


,case_id,source_group_id,case_family,case_variant,gold_label,f1_prediction,correct,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact
0,consolidation_lag_2sec_000,heldout_source_000,lag,lag_2sec,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
1,consolidation_lag_2sec_007,heldout_source_007,lag,lag_2sec,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
2,consolidation_lag_2sec_010,heldout_source_010,lag,lag_2sec,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
3,consolidation_lag_2sec_018,heldout_source_018,lag,lag_2sec,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
4,consolidation_lag_2sec_021,heldout_source_021,lag,lag_2sec,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,consolidation_wrong_partner_094,heldout_source_094,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
196,consolidation_wrong_partner_096,heldout_source_096,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
197,consolidation_wrong_partner_097,heldout_source_097,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,COMPATIBLE,TEMPORAL,True
198,consolidation_wrong_partner_098,heldout_source_098,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True


## 13. Construct the semantic pseudo-gold targets

The complete unified F1 prompt remains the model input.

The target output is the frozen F1 structured JSON with exactly one possible
replacement: `semantic_assessment`.


In [ ]:

# ============================================================
# TASK-DERIVED PSEUDO-GOLD SEMANTIC TARGETS
#
# NORMAL       -> COMPATIBLE
# LAG          -> COMPATIBLE
# WRONG_PARTNER:
#   frozen F1 INCOMPATIBLE -> INCOMPATIBLE
#   frozen F1 COMPATIBLE   -> INCOMPATIBLE
#   frozen F1 LIMITED      -> LIMITED
#
# These are pseudo-gold labels derived from the experimental
# construction and the agreed uncertainty policy. They are not
# human semantic annotations.
# ============================================================

from collections import OrderedDict

OUTPUT_FIELD_ORDER = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]

def pseudo_gold_semantic_target(case, teacher_record):
    family = get_case_family(case)

    if family in {
        "normal",
        "lag",
    }:
        return "COMPATIBLE"

    if family == "wrong_partner":
        baseline_semantic = teacher_record[
            "semantic_assessment"
        ]

        if baseline_semantic in {
            "INCOMPATIBLE",
            "COMPATIBLE",
        }:
            return "INCOMPATIBLE"

        if baseline_semantic == "LIMITED":
            return "LIMITED"

    raise ValueError(
        "No semantic pseudo-target is defined for "
        f"{family}. Silent cases are intentionally excluded."
    )

def build_training_row(case):
    case_id = str(case["case_id"])
    teacher = F1_RECORDS[case_id]

    semantic_target = pseudo_gold_semantic_target(
        case,
        teacher,
    )

    prompt, payload = build_semantic_ablation_prompt(
        case,
        F1_CONFIG,
    )

    target_object = OrderedDict()

    for field in OUTPUT_FIELD_ORDER:
        if field == "semantic_assessment":
            target_object[field] = semantic_target

        elif field == "label":
            target_object[field] = teacher["prediction"]

        else:
            target_object[field] = teacher[field]

    assert set(target_object) == set(REASONING_SCHEMA_KEYS)

    for field, value in target_object.items():
        assert value in REASONING_ALLOWED_VALUES[field], (
            f"Invalid target {field}={value} for {case_id}"
        )

    target_json = json.dumps(
        target_object,
        ensure_ascii=False,
        indent=2,
    )

    return {
        "case_id": case_id,
        "source_group_id": str(case["source_group_id"]),
        "case_family": get_case_family(case),
        "case_variant": str(case["case_variant"]),
        "gold_binary_label": str(case["gold_binary_label"]).upper(),
        "f1_prediction": teacher["prediction"],
        "f1_semantic_assessment": teacher["semantic_assessment"],
        "semantic_pseudo_gold": semantic_target,
        "prompt": prompt,
        "payload": payload,
        "target_object": dict(target_object),
        "target_json": target_json,
    }

TRAIN_ROWS = [
    build_training_row(case)
    for case in TRAIN_CASES
]

VALIDATION_ROWS = [
    build_training_row(case)
    for case in VALIDATION_CASES
]

assert len(TRAIN_ROWS) == 120
assert len(VALIDATION_ROWS) == 30

def rows_dataframe(rows):
    return pd.DataFrame([
        {
            key: row[key]
            for key in [
                "case_id",
                "source_group_id",
                "case_family",
                "case_variant",
                "gold_binary_label",
                "f1_prediction",
                "f1_semantic_assessment",
                "semantic_pseudo_gold",
            ]
        }
        for row in rows
    ])

TRAIN_TARGETS_DF = rows_dataframe(TRAIN_ROWS)
VALIDATION_TARGETS_DF = rows_dataframe(VALIDATION_ROWS)

print("=" * 100)
print("TRAINING PSEUDO-GOLD TARGETS")
print("=" * 100)

display(pd.crosstab(
    TRAIN_TARGETS_DF["case_family"],
    TRAIN_TARGETS_DF["semantic_pseudo_gold"],
    margins=True,
))

print("\nWrong-partner baseline semantic composition in training:")
display(
    TRAIN_TARGETS_DF[
        TRAIN_TARGETS_DF["case_family"] == "wrong_partner"
    ]["f1_semantic_assessment"]
    .value_counts()
    .rename_axis("f1_semantic_assessment")
    .reset_index(name="count")
)

actual_train_wp_counts = (
    TRAIN_TARGETS_DF[
        TRAIN_TARGETS_DF["case_family"] == "wrong_partner"
    ]["f1_semantic_assessment"]
    .value_counts()
    .to_dict()
)

assert actual_train_wp_counts == TRAIN_WP_BASELINE_COUNTS, (
    actual_train_wp_counts,
    TRAIN_WP_BASELINE_COUNTS,
)

print("\nValidation pseudo-gold targets:")
display(pd.crosstab(
    VALIDATION_TARGETS_DF["case_family"],
    VALIDATION_TARGETS_DF["semantic_pseudo_gold"],
    margins=True,
))


TRAINING PSEUDO-GOLD TARGETS


semantic_pseudo_gold,COMPATIBLE,INCOMPATIBLE,LIMITED,All
case_family,,,,
lag,40,0,0,40
normal,40,0,0,40
wrong_partner,0,30,10,40
All,80,30,10,120



Wrong-partner baseline semantic composition in training:


,f1_semantic_assessment,count
0,INCOMPATIBLE,16
1,COMPATIBLE,14
2,LIMITED,10



Validation pseudo-gold targets:


semantic_pseudo_gold,COMPATIBLE,INCOMPATIBLE,LIMITED,All
case_family,,,,
lag,10,0,0,10
normal,10,0,0,10
wrong_partner,0,7,3,10
All,20,7,3,30


## 14. Save the split and pseudo-annotation manifests

This makes the experiment reproducible and prevents accidental split changes.


In [ ]:

# ============================================================
# SAVE THE SPLIT AND PSEUDO-ANNOTATION MANIFEST
# ============================================================

FINE_TUNING_ROOT = (
    OUT_DIR
    / "structured_r1_f1_semantic_targeted_finetuning"
)

FINE_TUNING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SPLIT_MANIFEST_PATH = (
    FINE_TUNING_ROOT
    / "grouped_split_manifest.json"
)

TRAIN_TARGETS_PATH = (
    FINE_TUNING_ROOT
    / "training_pseudo_targets.json"
)

VALIDATION_TARGETS_PATH = (
    FINE_TUNING_ROOT
    / "validation_pseudo_targets.json"
)

split_manifest = {
    "split_seed": SPLIT_SEED,
    "train_group_ids": train_group_ids,
    "validation_group_ids": validation_group_ids,
    "test_group_ids": test_group_ids,
    "train_group_count": len(train_group_ids),
    "validation_group_count": len(validation_group_ids),
    "test_group_count": len(test_group_ids),
    "train_wrong_partner_baseline_counts": TRAIN_WP_BASELINE_COUNTS,
    "validation_wrong_partner_baseline_counts": VALIDATION_WP_BASELINE_COUNTS,
    "silent_cases_in_training": False,
    "f1_prompt_sha256": F1_CONFIG["reasoning_prompt_sha256"],
    "f1_cache_path": str(F1_CACHE_PATH),
    "pseudo_gold_policy": {
        "normal": "COMPATIBLE",
        "lag": "COMPATIBLE",
        "wrong_partner_f1_incompatible": "INCOMPATIBLE",
        "wrong_partner_f1_compatible": "INCOMPATIBLE",
        "wrong_partner_f1_limited": "LIMITED",
        "silent_partner": "EXCLUDED_FROM_TRAINING",
    },
}

SPLIT_MANIFEST_PATH.write_text(
    json.dumps(
        split_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

def serializable_rows(rows):
    return [
        {
            key: value
            for key, value in row.items()
            if key not in {
                "prompt",
                "payload",
            }
        }
        for row in rows
    ]

TRAIN_TARGETS_PATH.write_text(
    json.dumps(
        serializable_rows(TRAIN_ROWS),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

VALIDATION_TARGETS_PATH.write_text(
    json.dumps(
        serializable_rows(VALIDATION_ROWS),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Fine-tuning root:", FINE_TUNING_ROOT)
print("Split manifest:", SPLIT_MANIFEST_PATH)
print("Training targets:", TRAIN_TARGETS_PATH)
print("Validation targets:", VALIDATION_TARGETS_PATH)


Fine-tuning root: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning
Split manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/grouped_split_manifest.json
Training targets: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/training_pseudo_targets.json
Validation targets: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/validation_pseudo_targets.json


## 15. Manually inspect representative targets

The cell prints:

- one NORMAL target;
- one LAG target;
- a wrong-partner `COMPATIBLE → INCOMPATIBLE` correction;
- an already-correct wrong-partner `INCOMPATIBLE`;
- a wrong-partner `LIMITED → LIMITED`;
- the complete exact F1 prompt for one hard correction.

Do not start training before checking this output.


In [ ]:

# ============================================================
# MANUAL INSPECTION OF REPRESENTATIVE TRAINING TARGETS
# ============================================================

def first_row_matching(
    rows,
    *,
    family,
    f1_semantic=None,
):
    for row in rows:
        if row["case_family"] != family:
            continue

        if (
            f1_semantic is not None
            and row["f1_semantic_assessment"]
            != f1_semantic
        ):
            continue

        return row

    raise ValueError(
        f"No row found for {family} / {f1_semantic}"
    )

inspection_rows = [
    first_row_matching(
        TRAIN_ROWS,
        family="normal",
    ),
    first_row_matching(
        TRAIN_ROWS,
        family="lag",
    ),
    first_row_matching(
        TRAIN_ROWS,
        family="wrong_partner",
        f1_semantic="COMPATIBLE",
    ),
    first_row_matching(
        TRAIN_ROWS,
        family="wrong_partner",
        f1_semantic="INCOMPATIBLE",
    ),
    first_row_matching(
        TRAIN_ROWS,
        family="wrong_partner",
        f1_semantic="LIMITED",
    ),
]

for row in inspection_rows:
    print("\n" + "=" * 100)
    print(
        row["case_family"],
        "| F1 semantic:",
        row["f1_semantic_assessment"],
        "| pseudo-gold:",
        row["semantic_pseudo_gold"],
    )
    print("=" * 100)
    print(row["target_json"])

print("\nExact F1 prompt for one hard wrong-partner correction:")
hard_wp_row = first_row_matching(
    TRAIN_ROWS,
    family="wrong_partner",
    f1_semantic="COMPATIBLE",
)

print(hard_wp_row["prompt"])



normal | F1 semantic: COMPATIBLE | pseudo-gold: COMPATIBLE
{
  "participation_assessment": "VALID",
  "local_temporal_assessment": "NORMAL",
  "global_temporal_assessment": "NORMAL",
  "temporal_assessment": "NORMAL",
  "semantic_assessment": "COMPATIBLE",
  "decisive_dimension": "SEMANTIC",
  "label": "NORMAL"
}

lag | F1 semantic: COMPATIBLE | pseudo-gold: COMPATIBLE
{
  "participation_assessment": "VALID",
  "local_temporal_assessment": "ANOMALOUS",
  "global_temporal_assessment": "ANOMALOUS",
  "temporal_assessment": "ANOMALOUS",
  "semantic_assessment": "COMPATIBLE",
  "decisive_dimension": "TEMPORAL",
  "label": "ANOMALOUS"
}

wrong_partner | F1 semantic: COMPATIBLE | pseudo-gold: INCOMPATIBLE
{
  "participation_assessment": "VALID",
  "local_temporal_assessment": "ANOMALOUS",
  "global_temporal_assessment": "ANOMALOUS",
  "temporal_assessment": "ANOMALOUS",
  "semantic_assessment": "INCOMPATIBLE",
  "decisive_dimension": "TEMPORAL",
  "label": "ANOMALOUS"
}

wrong_partner | F1 

## 16. Load Qwen2.5-Omni Thinker for QLoRA

The Thinker is used because the task is text-only over the already consolidated
evidence packet. The notebook uses 4-bit NF4 quantisation by default to reduce
GPU memory requirements.


In [ ]:

# ============================================================
# LOAD QWEN2.5-OMNI THINKER FOR LIGHTWEIGHT QLORA FINE-TUNING
# ============================================================

import gc
import torch

from transformers import (
    BitsAndBytesConfig,
    Qwen2_5OmniProcessor,
    Qwen2_5OmniThinkerForConditionalGeneration,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

assert torch.cuda.is_available(), (
    "A CUDA GPU runtime is required."
)

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

USE_4BIT = True

MODEL_SOURCE = (
    str(MODEL_PATH)
    if MODEL_PATH.exists()
    else MODEL_ID
)

quantization_config = None

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_SOURCE
)

tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

assert tokenizer.is_fast, (
    "A fast tokenizer is required for exact semantic token-span masking."
)

model = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_SOURCE,
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("Loaded model source:", MODEL_SOURCE)
print("Compute dtype:", COMPUTE_DTYPE)
print("4-bit QLoRA:", USE_4BIT)


[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_k.bias                                  | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.weight                             | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.bias                               | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn_norm.linear.bias                           | UNEXPECTED |  | 
talker.model.laye

Loaded model source: /content/drive/MyDrive/Qwen2.5-Omni-7B
Compute dtype: torch.bfloat16
4-bit QLoRA: True


In [ ]:
print("Model class:", type(model).__name__)
print("Model source:", MODEL_SOURCE)
print("Tokenizer vocabulary:", len(tokenizer))
print(
    "Text embedding vocabulary:",
    model.get_input_embeddings().num_embeddings,
)

assert type(model).__name__ == (
    "Qwen2_5OmniThinkerForConditionalGeneration"
)

assert str(MODEL_SOURCE) == (
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)

assert (
    model.get_input_embeddings().num_embeddings
    >= len(tokenizer)
), (
    "Tokenizer/model vocabulary mismatch."
)

print("\nSame Qwen2.5-Omni-7B Thinker checkpoint confirmed.")
print(
    "Important: base weights are loaded in 4-bit, "
    "so inference is not bit-identical to the original BF16 F1."
)

Model class: Qwen2_5OmniThinkerForConditionalGeneration
Model source: /content/drive/MyDrive/Qwen2.5-Omni-7B
Tokenizer vocabulary: 151665
Text embedding vocabulary: 152064

Same Qwen2.5-Omni-7B Thinker checkpoint confirmed.
Important: base weights are loaded in 4-bit, so inference is not bit-identical to the original BF16 F1.


## 17. Attach a lightweight LoRA adapter to the text-language stack

The notebook discovers the exact language projection modules at runtime and
excludes audio, vision, Talker, and token-to-wave modules.


In [ ]:

# ============================================================
# TARGET ONLY THE TEXT-LANGUAGE TRANSFORMER PROJECTIONS
#
# The exact module names are discovered from the loaded model.
# Audio/vision modules are excluded because this experiment is
# text-only over the already consolidated F1 evidence packet.
# ============================================================

PREFERRED_LORA_SUFFIXES = (
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
)

EXCLUDED_MODULE_MARKERS = (
    "audio",
    "vision",
    "visual",
    "talker",
    "token2wav",
)

def discover_text_lora_target_modules(model):
    linear_module_names = [
        name
        for name, module in model.named_modules()
        if isinstance(module, torch.nn.Linear)
    ]

    candidates = [
        name
        for name in linear_module_names
        if name.endswith(PREFERRED_LORA_SUFFIXES)
    ]

    text_candidates = [
        name
        for name in candidates
        if not any(
            marker in name.lower()
            for marker in EXCLUDED_MODULE_MARKERS
        )
    ]

    language_marked = [
        name
        for name in text_candidates
        if (
            "language_model" in name.lower()
            or ".model.layers." in name.lower()
        )
    ]

    if language_marked:
        text_candidates = language_marked

    assert text_candidates, (
        "No text-language LoRA targets were discovered."
    )

    return sorted(set(text_candidates))

LORA_TARGET_MODULES = discover_text_lora_target_modules(
    model
)

print("=" * 100)
print("DISCOVERED TEXT-LANGUAGE LORA TARGET MODULES")
print("=" * 100)
print("Number of target modules:", len(LORA_TARGET_MODULES))
print("First 20 targets:")

for name in LORA_TARGET_MODULES[:20]:
    print(" ", name)

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()


DISCOVERED TEXT-LANGUAGE LORA TARGET MODULES
Number of target modules: 196
First 20 targets:
  model.layers.0.mlp.down_proj
  model.layers.0.mlp.gate_proj
  model.layers.0.mlp.up_proj
  model.layers.0.self_attn.k_proj
  model.layers.0.self_attn.o_proj
  model.layers.0.self_attn.q_proj
  model.layers.0.self_attn.v_proj
  model.layers.1.mlp.down_proj
  model.layers.1.mlp.gate_proj
  model.layers.1.mlp.up_proj
  model.layers.1.self_attn.k_proj
  model.layers.1.self_attn.o_proj
  model.layers.1.self_attn.q_proj
  model.layers.1.self_attn.v_proj
  model.layers.10.mlp.down_proj
  model.layers.10.mlp.gate_proj
  model.layers.10.mlp.up_proj
  model.layers.10.self_attn.k_proj
  model.layers.10.self_attn.o_proj
  model.layers.10.self_attn.q_proj
trainable params: 20,185,088 || all params: 8,951,998,976 || trainable%: 0.2255


## 18. Tokenise the exact prompt and construct the two loss masks

For every training sequence:

- prompt tokens: zero loss;
- `semantic_assessment` value tokens: semantic pseudo-gold loss;
- all other assistant tokens: frozen-F1 preservation loss.

The two components are independently normalised before applying
`PRESERVATION_LAMBDA`.


In [ ]:

# ============================================================
# TOKENIZE THE FULL UNIFIED F1 PROMPT AND WEIGHT THE OUTPUT
#
# Loss definition:
#
#   L_total = L_semantic_gold + λ * L_preservation
#
# L_semantic_gold:
#   token-level CE only on the semantic_assessment VALUE.
#
# L_preservation:
#   token-level CE against the frozen F1 teacher output on every
#   other assistant-output token: schema, participation, temporal
#   assessments, decisive_dimension and final label.
#
# Prompt tokens receive zero loss.
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

MAX_TRAIN_SEQUENCE_TOKENS = 16384
PRESERVATION_LAMBDA = 0.50

def build_prompt_token_ids(prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                }
            ],
        }
    ]

    prompt_inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": False,
        },
    )

    return prompt_inputs["input_ids"][0].tolist()

def tokenize_target_with_semantic_span(
    target_json,
    semantic_target,
):
    needle = (
        '"semantic_assessment": '
        f'"{semantic_target}"'
    )

    assert target_json.count(needle) == 1

    needle_start = target_json.index(needle)

    semantic_start = (
        needle_start
        + needle.index(semantic_target)
    )

    semantic_end = (
        semantic_start
        + len(semantic_target)
    )

    encoded = tokenizer(
        target_json,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    target_ids = list(encoded["input_ids"])
    offsets = list(encoded["offset_mapping"])

    semantic_weights = []
    preservation_weights = []

    for start, end in offsets:
        overlaps_semantic = (
            max(start, semantic_start)
            <
            min(end, semantic_end)
        )

        semantic_weights.append(
            1.0 if overlaps_semantic else 0.0
        )

        preservation_weights.append(
            0.0 if overlaps_semantic else 1.0
        )

    assert sum(semantic_weights) > 0, (
        "Semantic value did not align to any token."
    )

    eos_token_id = tokenizer.eos_token_id

    if eos_token_id is not None:
        target_ids.append(int(eos_token_id))
        semantic_weights.append(0.0)
        preservation_weights.append(1.0)

    return (
        target_ids,
        semantic_weights,
        preservation_weights,
    )

def tokenize_training_row(row):
    prompt_ids = build_prompt_token_ids(
        row["prompt"]
    )

    (
        target_ids,
        semantic_target_weights,
        preservation_target_weights,
    ) = tokenize_target_with_semantic_span(
        row["target_json"],
        row["semantic_pseudo_gold"],
    )

    input_ids = prompt_ids + target_ids

    assert len(input_ids) <= MAX_TRAIN_SEQUENCE_TOKENS, (
        f"Sequence too long for {row['case_id']}: "
        f"{len(input_ids)} tokens."
    )

    prompt_zeros = [0.0] * len(prompt_ids)

    semantic_weights = (
        prompt_zeros
        + semantic_target_weights
    )

    preservation_weights = (
        prompt_zeros
        + preservation_target_weights
    )

    assert len(input_ids) == len(semantic_weights)
    assert len(input_ids) == len(preservation_weights)

    return {
        "case_id": row["case_id"],
        "case_family": row["case_family"],
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "semantic_weights": semantic_weights,
        "preservation_weights": preservation_weights,
        "sequence_length": len(input_ids),
        "semantic_token_count": int(sum(semantic_weights)),
        "preservation_token_count": int(sum(preservation_weights)),
    }

TOKENIZED_TRAIN_ROWS = [
    tokenize_training_row(row)
    for row in TRAIN_ROWS
]

TOKENIZED_VALIDATION_ROWS = [
    tokenize_training_row(row)
    for row in VALIDATION_ROWS
]

lengths = [
    row["sequence_length"]
    for row in (
        TOKENIZED_TRAIN_ROWS
        + TOKENIZED_VALIDATION_ROWS
    )
]

print("=" * 100)
print("TOKENIZATION AUDIT")
print("=" * 100)
print("Minimum sequence length:", min(lengths))
print("Median sequence length:", int(np.median(lengths)))
print("Maximum sequence length:", max(lengths))

display(pd.DataFrame([
    {
        "case_id": row["case_id"],
        "case_family": row["case_family"],
        "sequence_length": row["sequence_length"],
        "semantic_token_count": row["semantic_token_count"],
        "preservation_token_count": row["preservation_token_count"],
    }
    for row in TOKENIZED_TRAIN_ROWS
]).head(12))


TOKENIZATION AUDIT
Minimum sequence length: 5271
Median sequence length: 5378
Maximum sequence length: 5500


,case_id,case_family,sequence_length,semantic_token_count,preservation_token_count
0,consolidation_lag_2sec_001,lag,5358,3,84
1,consolidation_lag_2sec_003,lag,5368,3,72
2,consolidation_lag_2sec_009,lag,5370,3,84
3,consolidation_lag_2sec_012,lag,5360,3,84
4,consolidation_lag_2sec_013,lag,5408,3,84
5,consolidation_lag_2sec_014,lag,5343,3,72
6,consolidation_lag_2sec_015,lag,5384,3,84
7,consolidation_lag_2sec_017,lag,5500,3,84
8,consolidation_lag_2sec_025,lag,5382,3,72
9,consolidation_lag_2sec_030,lag,5370,3,84


In [ ]:
all_tokenized_rows = (
    TOKENIZED_TRAIN_ROWS
    + TOKENIZED_VALIDATION_ROWS
)

max_sequence_length = max(
    row["sequence_length"]
    for row in all_tokenized_rows
)

print("Maximum sequence length:", max_sequence_length)
print("Maximum allowed length:", MAX_TRAIN_SEQUENCE_TOKENS)

assert (
    max_sequence_length
    <= MAX_TRAIN_SEQUENCE_TOKENS
), (
    f"Sequence length {max_sequence_length} exceeds "
    f"the limit {MAX_TRAIN_SEQUENCE_TOKENS}."
)

print("All sequences fit without truncation.")

Maximum sequence length: 5500
Maximum allowed length: 16384
All sequences fit without truncation.


## 19. Build the weighted PyTorch datasets


In [ ]:

# ============================================================
# PYTORCH DATASET AND DYNAMIC PADDING
# ============================================================

class WeightedReasoningDataset(Dataset):
    def __init__(self, tokenized_rows):
        self.rows = list(tokenized_rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

def weighted_reasoning_collator(batch):
    max_length = max(
        len(item["input_ids"])
        for item in batch
    )

    pad_token_id = int(tokenizer.pad_token_id)

    input_ids = []
    attention_mask = []
    semantic_weights = []
    preservation_weights = []

    metadata = []

    for item in batch:
        pad_length = (
            max_length
            - len(item["input_ids"])
        )

        input_ids.append(
            item["input_ids"]
            + [pad_token_id] * pad_length
        )

        attention_mask.append(
            item["attention_mask"]
            + [0] * pad_length
        )

        semantic_weights.append(
            item["semantic_weights"]
            + [0.0] * pad_length
        )

        preservation_weights.append(
            item["preservation_weights"]
            + [0.0] * pad_length
        )

        metadata.append({
            "case_id": item["case_id"],
            "case_family": item["case_family"],
        })

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            attention_mask,
            dtype=torch.long,
        ),
        "semantic_weights": torch.tensor(
            semantic_weights,
            dtype=torch.float32,
        ),
        "preservation_weights": torch.tensor(
            preservation_weights,
            dtype=torch.float32,
        ),
        "metadata": metadata,
    }

TRAIN_DATASET = WeightedReasoningDataset(
    TOKENIZED_TRAIN_ROWS
)

VALIDATION_DATASET = WeightedReasoningDataset(
    TOKENIZED_VALIDATION_ROWS
)


## 20. Verify the weighted loss with a forward-only smoke test


In [ ]:

# ============================================================
# WEIGHTED LOSS IMPLEMENTATION
# ============================================================

import torch.nn.functional as F

def compute_weighted_reasoning_loss(
    model,
    batch,
    preservation_lambda=PRESERVATION_LAMBDA,
):
    device = next(
        parameter.device
        for parameter in model.parameters()
        if parameter.is_cuda
    )

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    semantic_weights = (
        batch["semantic_weights"]
        .to(device)
    )

    preservation_weights = (
        batch["preservation_weights"]
        .to(device)
    )

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True,
    )

    logits = outputs.logits

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    shift_semantic_weights = (
        semantic_weights[:, 1:]
        .contiguous()
    )

    shift_preservation_weights = (
        preservation_weights[:, 1:]
        .contiguous()
    )

    token_losses = F.cross_entropy(
        shift_logits.view(
            -1,
            shift_logits.size(-1),
        ),
        shift_labels.view(-1),
        reduction="none",
    ).view_as(shift_labels)

    semantic_denominator = (
        shift_semantic_weights.sum()
        .clamp_min(1.0)
    )

    preservation_denominator = (
        shift_preservation_weights.sum()
        .clamp_min(1.0)
    )

    semantic_loss = (
        token_losses
        * shift_semantic_weights
    ).sum() / semantic_denominator

    preservation_loss = (
        token_losses
        * shift_preservation_weights
    ).sum() / preservation_denominator

    total_loss = (
        semantic_loss
        + preservation_lambda
        * preservation_loss
    )

    return {
        "loss": total_loss,
        "semantic_loss": semantic_loss.detach(),
        "preservation_loss": preservation_loss.detach(),
    }

# One forward-only smoke test before training.
SMOKE_TEST_LOADER = DataLoader(
    TRAIN_DATASET,
    batch_size=1,
    shuffle=False,
    collate_fn=weighted_reasoning_collator,
)

smoke_batch = next(iter(SMOKE_TEST_LOADER))

model.eval()

with torch.no_grad():
    smoke_losses = compute_weighted_reasoning_loss(
        model,
        smoke_batch,
    )

print("=" * 100)
print("WEIGHTED LOSS SMOKE TEST")
print("=" * 100)
print(
    "Total:",
    float(smoke_losses["loss"].cpu()),
)
print(
    "Semantic:",
    float(smoke_losses["semantic_loss"].cpu()),
)
print(
    "Preservation:",
    float(smoke_losses["preservation_loss"].cpu()),
)

model.train()


WEIGHTED LOSS SMOKE TEST
Total: 0.021483052521944046
Semantic: 0.012248600833117962
Preservation: 0.01846890151500702


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5OmniThinkerForConditionalGeneration(
      (audio_tower): Qwen2_5OmniAudioEncoder(
        (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
        (positional_embedding): SinusoidsPositionEmbedding()
        (audio_bos_eos_token): Embedding(2, 3584)
        (layers): ModuleList(
          (0-31): 32 x Qwen2_5OmniAudioEncoderLayer(
            (self_attn): Qwen2_5OmniAudioAttention(
              (k_proj): Linear4bit(in_features=1280, out_features=1280, bias=False)
              (v_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              (q_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              (out_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
 

## 21. Run targeted QLoRA fine-tuning

Default conservative settings:

- LoRA rank: 8
- LoRA alpha: 16
- learning rate: 2e-5
- gradient accumulation: 4
- maximum epochs: 3
- early stopping patience: 1
- preservation coefficient λ: 0.50

The best adapter is selected using validation
`L_semantic + λ·L_preservation`.


In [ ]:

# ============================================================
# LIGHTWEIGHT TARGETED QLORA TRAINING
# ============================================================

import math
import time

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 1

TRAINING_RUN_DIR = (
    FINE_TUNING_ROOT
    / (
        f"qlora_r{LORA_RANK}_"
        f"lambda_{str(PRESERVATION_LAMBDA).replace('.', '_')}_"
        f"seed_{SPLIT_SEED}"
    )
)

BEST_ADAPTER_DIR = (
    TRAINING_RUN_DIR
    / "best_adapter"
)

LATEST_ADAPTER_DIR = (
    TRAINING_RUN_DIR
    / "latest_adapter"
)

TRAINING_HISTORY_PATH = (
    TRAINING_RUN_DIR
    / "training_history.json"
)

TRAINING_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.manual_seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

train_generator = torch.Generator()
train_generator.manual_seed(SPLIT_SEED)

train_loader = DataLoader(
    TRAIN_DATASET,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    collate_fn=weighted_reasoning_collator,
)

validation_loader = DataLoader(
    VALIDATION_DATASET,
    batch_size=1,
    shuffle=False,
    collate_fn=weighted_reasoning_collator,
)

optimizer = AdamW(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader)
    / GRADIENT_ACCUMULATION_STEPS
)

total_update_steps = (
    MAX_EPOCHS
    * updates_per_epoch
)

warmup_steps = max(
    1,
    int(
        total_update_steps
        * WARMUP_RATIO
    ),
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

def evaluate_teacher_forced_loss(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_semantic_loss = 0.0
    total_preservation_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in data_loader:
            losses = compute_weighted_reasoning_loss(
                model,
                batch,
            )

            total_loss += float(
                losses["loss"].cpu()
            )

            total_semantic_loss += float(
                losses["semantic_loss"].cpu()
            )

            total_preservation_loss += float(
                losses["preservation_loss"].cpu()
            )

            num_batches += 1

    model.train()

    return {
        "loss": total_loss / max(num_batches, 1),
        "semantic_loss": (
            total_semantic_loss
            / max(num_batches, 1)
        ),
        "preservation_loss": (
            total_preservation_loss
            / max(num_batches, 1)
        ),
    }

RUN_TRAINING = True

training_history = []
best_validation_loss = float("inf")
epochs_without_improvement = 0

if RUN_TRAINING:
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for epoch_index in range(MAX_EPOCHS):
        epoch_number = epoch_index + 1
        epoch_started = time.perf_counter()

        running_total = 0.0
        running_semantic = 0.0
        running_preservation = 0.0
        batch_count = 0

        for step_index, batch in enumerate(
            train_loader,
            start=1,
        ):
            losses = compute_weighted_reasoning_loss(
                model,
                batch,
            )

            scaled_loss = (
                losses["loss"]
                / GRADIENT_ACCUMULATION_STEPS
            )

            scaled_loss.backward()

            running_total += float(
                losses["loss"].detach().cpu()
            )

            running_semantic += float(
                losses["semantic_loss"].cpu()
            )

            running_preservation += float(
                losses["preservation_loss"].cpu()
            )

            batch_count += 1

            should_update = (
                step_index
                % GRADIENT_ACCUMULATION_STEPS
                == 0
                or step_index == len(train_loader)
            )

            if should_update:
                torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in model.parameters()
                        if parameter.requires_grad
                    ],
                    MAX_GRAD_NORM,
                )

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        validation_metrics = (
            evaluate_teacher_forced_loss(
                model,
                validation_loader,
            )
        )

        epoch_seconds = (
            time.perf_counter()
            - epoch_started
        )

        epoch_record = {
            "epoch": epoch_number,
            "train_loss": (
                running_total
                / max(batch_count, 1)
            ),
            "train_semantic_loss": (
                running_semantic
                / max(batch_count, 1)
            ),
            "train_preservation_loss": (
                running_preservation
                / max(batch_count, 1)
            ),
            "validation_loss": validation_metrics["loss"],
            "validation_semantic_loss": (
                validation_metrics["semantic_loss"]
            ),
            "validation_preservation_loss": (
                validation_metrics["preservation_loss"]
            ),
            "learning_rate": scheduler.get_last_lr()[0],
            "epoch_seconds": round(epoch_seconds, 3),
        }

        training_history.append(epoch_record)

        print("\n" + "=" * 100)
        print(f"EPOCH {epoch_number}")
        print("=" * 100)
        print(json.dumps(
            epoch_record,
            indent=2,
        ))

        LATEST_ADAPTER_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            LATEST_ADAPTER_DIR
        )

        processor.save_pretrained(
            LATEST_ADAPTER_DIR
        )

        if (
            validation_metrics["loss"]
            < best_validation_loss
        ):
            best_validation_loss = (
                validation_metrics["loss"]
            )

            epochs_without_improvement = 0

            BEST_ADAPTER_DIR.mkdir(
                parents=True,
                exist_ok=True,
            )

            model.save_pretrained(
                BEST_ADAPTER_DIR
            )

            processor.save_pretrained(
                BEST_ADAPTER_DIR
            )

            print(
                "Saved new best adapter:",
                BEST_ADAPTER_DIR,
            )

        else:
            epochs_without_improvement += 1

        TRAINING_HISTORY_PATH.write_text(
            json.dumps(
                training_history,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "Early stopping triggered."
            )
            break

else:
    assert BEST_ADAPTER_DIR.exists(), (
        "RUN_TRAINING=False but no saved best adapter exists."
    )

print("\nBest adapter:", BEST_ADAPTER_DIR)
print("Best validation loss:", best_validation_loss)



EPOCH 1
{
  "epoch": 1,
  "train_loss": 0.24751303675972547,
  "train_semantic_loss": 0.23726116514056533,
  "train_preservation_loss": 0.02050373995443806,
  "validation_loss": 0.18200972235451143,
  "validation_semantic_loss": 0.17365625047823413,
  "validation_preservation_loss": 0.01670694719068706,
  "learning_rate": 1.4814814814814815e-05,
  "epoch_seconds": 918.179
}
Saved new best adapter: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/best_adapter

EPOCH 2
{
  "epoch": 2,
  "train_loss": 0.13520103956495103,
  "train_semantic_loss": 0.1282455870488775,
  "train_preservation_loss": 0.013910905706385772,
  "validation_loss": 0.12719482691027223,
  "validation_semantic_loss": 0.12017349008237943,
  "validation_preservation_loss": 0.014042672344172995,
  "learning_rate": 7.4074074074074075e-06,
  "epoch_seconds": 916.64
}
Saved new best adapter: /content/drive/MyDrive/qwen_vad_turns_norm

## 22. Reload the best validation adapter


In [ ]:

# ============================================================
# RELOAD THE BEST ADAPTER FOR GENERATIVE EVALUATION
# ============================================================

from peft import PeftModel

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model_for_evaluation = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_SOURCE,
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
)

model = PeftModel.from_pretrained(
    base_model_for_evaluation,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

print("Best fine-tuned adapter loaded for evaluation.")


Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_k.bias                                  | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.weight                             | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.bias                               | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn_norm.linear.bias                           | UNEXPECTED |  | 
talker.model.laye

Best fine-tuned adapter loaded for evaluation.


In [ ]:
# ============================================================
# RESUME FROM SAVED BEST ADAPTER — NO TRAINING
# Run after Sections 1–15 in a fresh Colab runtime.
# ============================================================

import gc
import torch

from transformers import (
    BitsAndBytesConfig,
    Qwen2_5OmniProcessor,
    Qwen2_5OmniThinkerForConditionalGeneration,
)

from peft import PeftModel

assert torch.cuda.is_available(), (
    "A CUDA GPU runtime is required."
)

# ------------------------------------------------------------
# SAME BASE MODEL AND QUANTISATION AS TRAINING
# ------------------------------------------------------------

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

USE_4BIT = True

MODEL_SOURCE = (
    str(MODEL_PATH)
    if MODEL_PATH.exists()
    else MODEL_ID
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_SOURCE
)

tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# SAME EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

LORA_RANK = 8
LORA_ALPHA = 16
PRESERVATION_LAMBDA = 0.50
LEARNING_RATE = 2e-5

TRAINING_RUN_DIR = (
    FINE_TUNING_ROOT
    / (
        f"qlora_r{LORA_RANK}_"
        f"lambda_{str(PRESERVATION_LAMBDA).replace('.', '_')}_"
        f"seed_{SPLIT_SEED}"
    )
)

BEST_ADAPTER_DIR = (
    TRAINING_RUN_DIR
    / "best_adapter"
)

TRAINING_HISTORY_PATH = (
    TRAINING_RUN_DIR
    / "training_history.json"
)

FINE_TUNED_TEST_CACHE_PATH = (
    TRAINING_RUN_DIR
    / "held_out_test_predictions_cache.json"
)

# ------------------------------------------------------------
# VERIFY THAT TRAINING OUTPUTS EXIST
# ------------------------------------------------------------

assert BEST_ADAPTER_DIR.exists(), (
    f"Best adapter directory not found:\n{BEST_ADAPTER_DIR}"
)

assert (
    BEST_ADAPTER_DIR / "adapter_config.json"
).exists(), (
    "adapter_config.json was not found."
)

adapter_weight_candidates = [
    BEST_ADAPTER_DIR / "adapter_model.safetensors",
    BEST_ADAPTER_DIR / "adapter_model.bin",
]

assert any(
    path.exists()
    for path in adapter_weight_candidates
), (
    "No saved LoRA adapter weights were found."
)

print("Saved adapter found:")
print(BEST_ADAPTER_DIR)

if TRAINING_HISTORY_PATH.exists():
    training_history = json.loads(
        TRAINING_HISTORY_PATH.read_text(
            encoding="utf-8"
        )
    )

    print(
        "Saved training epochs:",
        len(training_history),
    )

    print(
        "Best saved validation loss:",
        min(
            row["validation_loss"]
            for row in training_history
        ),
    )

# ------------------------------------------------------------
# LOAD BASE MODEL + SAVED BEST LORA ADAPTER
# ------------------------------------------------------------

if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model_for_evaluation = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_SOURCE,
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
)

model = PeftModel.from_pretrained(
    base_model_for_evaluation,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

print("\nBest fine-tuned adapter loaded.")
print("Training will NOT run again.")
print("Evaluation cache:", FINE_TUNED_TEST_CACHE_PATH)

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Saved adapter found:
/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/best_adapter
Saved training epochs: 3
Best saved validation loss: 0.11825642825569957


Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.convs1.{0, 1, 2}.weight                              | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
talker.model.layers.{0...23}.mlp.down_proj.weight                                                        | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_out.0.bias                              | UNEXPECTED |  | 
talker.model.layers.{0...23}.mlp.up_proj.weight                                                          | UNEXPECTED |  | 
token2wav.code2wa


Best fine-tuned adapter loaded.
Training will NOT run again.
Evaluation cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/held_out_test_predictions_cache.json


## 23. Evaluate on every remaining case

The 200 held-out cases contain:

- 50 NORMAL
- 50 LAG
- 50 WRONG_PARTNER
- 50 SILENT_PARTNER

Generation uses the exact frozen F1 prompt and the same deterministic decoder.
Each result is checkpointed to Google Drive.


In [ ]:

# ============================================================
# GENERATIVE EVALUATION ON ALL 200 HELD-OUT CASES
# ============================================================

from datetime import datetime, timezone
from tqdm.auto import tqdm

FINE_TUNED_TEST_CACHE_PATH = (
    TRAINING_RUN_DIR
    / "held_out_test_predictions_cache.json"
)

def utc_now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()

def atomic_write_json(path, value):
    temporary_path = Path(
        str(path) + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)

adapter_manifest_hash = sha256_text(
    json.dumps(
        {
            "best_adapter_dir": str(BEST_ADAPTER_DIR),
            "split_manifest": split_manifest,
            "preservation_lambda": PRESERVATION_LAMBDA,
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "learning_rate": LEARNING_RATE,
        },
        sort_keys=True,
        ensure_ascii=False,
    )
)

expected_test_cache_header = {
    "experiment": (
        "F1_semantic_targeted_QLoRA_held_out_test"
    ),
    "f1_prompt_sha256": (
        F1_CONFIG["reasoning_prompt_sha256"]
    ),
    "adapter_manifest_hash": adapter_manifest_hash,
    "test_group_ids": test_group_ids,
    "max_new_tokens": MAX_NEW_TOKENS_REASONING,
}

if FINE_TUNED_TEST_CACHE_PATH.exists():
    FINE_TUNED_TEST_CACHE = json.loads(
        FINE_TUNED_TEST_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )

    for key, expected_value in (
        expected_test_cache_header.items()
    ):
        assert (
            FINE_TUNED_TEST_CACHE[key]
            == expected_value
        ), (
            f"Incompatible evaluation cache: {key}"
        )

else:
    FINE_TUNED_TEST_CACHE = {
        **expected_test_cache_header,
        "created_at_utc": utc_now_iso(),
        "updated_at_utc": utc_now_iso(),
        "records": {},
    }

    atomic_write_json(
        FINE_TUNED_TEST_CACHE_PATH,
        FINE_TUNED_TEST_CACHE,
    )

for case in tqdm(
    TEST_CASES,
    desc="F1 semantic-targeted QLoRA test",
):
    case_id = str(case["case_id"])

    existing = (
        FINE_TUNED_TEST_CACHE[
            "records"
        ].get(case_id)
    )

    if (
        existing is not None
        and existing.get("prediction") in LABELS
        and existing.get("schema_exact") is True
    ):
        continue

    prompt, payload = build_semantic_ablation_prompt(
        case,
        F1_CONFIG,
    )

    prompt_hash = sha256_text(prompt)
    payload_hash = sha256_text(
        canonical_json(payload)
    )

    started = time.perf_counter()

    try:
        raw_output, input_token_count = (
            qwen_text_only_binary(
                prompt,
                max_new_tokens=(
                    MAX_NEW_TOKENS_REASONING
                ),
            )
        )

        parsed = parse_structured_reasoning_prediction(
            raw_output
        )

        generation_error = None

    except Exception as exc:
        raw_output = ""
        input_token_count = None

        parsed = {
            "prediction": None,
            "schema_exact": False,
            "participation_assessment": None,
            "local_temporal_assessment": None,
            "global_temporal_assessment": None,
            "temporal_assessment": None,
            "semantic_assessment": None,
            "decisive_dimension": None,
            "schema_errors": [],
            "parsed_output": None,
            "parse_mode": "generation_error",
        }

        generation_error = (
            f"{type(exc).__name__}: {exc}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed = time.perf_counter() - started

    FINE_TUNED_TEST_CACHE[
        "records"
    ][case_id] = {
        "case_id": case_id,
        "source_group_id": str(
            case["source_group_id"]
        ),
        "case_family": get_case_family(case),
        "case_variant": str(case["case_variant"]),
        "gold_binary_label": str(
            case["gold_binary_label"]
        ).upper(),
        "prompt_sha256": prompt_hash,
        "input_payload_sha256": payload_hash,
        "input_token_count": input_token_count,
        "raw_output": raw_output,
        "prediction": parsed["prediction"],
        "participation_assessment": (
            parsed["participation_assessment"]
        ),
        "local_temporal_assessment": (
            parsed["local_temporal_assessment"]
        ),
        "global_temporal_assessment": (
            parsed["global_temporal_assessment"]
        ),
        "temporal_assessment": (
            parsed["temporal_assessment"]
        ),
        "semantic_assessment": (
            parsed["semantic_assessment"]
        ),
        "decisive_dimension": (
            parsed["decisive_dimension"]
        ),
        "schema_exact": bool(
            parsed["schema_exact"]
        ),
        "schema_errors": parsed["schema_errors"],
        "parse_mode": parsed["parse_mode"],
        "parsed_output": parsed["parsed_output"],
        "generation_error": generation_error,
        "elapsed_seconds": round(elapsed, 4),
        "completed_at_utc": utc_now_iso(),
    }

    FINE_TUNED_TEST_CACHE[
        "updated_at_utc"
    ] = utc_now_iso()

    atomic_write_json(
        FINE_TUNED_TEST_CACHE_PATH,
        FINE_TUNED_TEST_CACHE,
    )

print("Held-out predictions:", len(
    FINE_TUNED_TEST_CACHE["records"]
))
print("Cache:", FINE_TUNED_TEST_CACHE_PATH)


F1 semantic-targeted QLoRA test:   0%|          | 0/200 [00:00<?, ?it/s]

Held-out predictions: 200
Cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/held_out_test_predictions_cache.json


## 24. Paired held-out comparison against frozen F1

The comparison reports:

- confusion matrices;
- accuracy, balanced accuracy, and macro-F1;
- per-family accuracy;
- semantic distributions per family;
- non-semantic branch stability;
- exact McNemar test;
- logical-consistency violations;
- per-case prediction and assessment changes.


In [ ]:

# ============================================================
# FINAL PAIRED COMPARISON:
# FROZEN F1 vs FINE-TUNED F1 ON THE SAME 200 TEST CASES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from scipy.stats import binomtest

def cache_records_dataframe(records):
    return pd.DataFrame(
        list(records.values())
    )

fine_tuned_df = cache_records_dataframe(
    FINE_TUNED_TEST_CACHE["records"]
)

test_case_ids = {
    str(case["case_id"])
    for case in TEST_CASES
}

frozen_f1_test_df = pd.DataFrame([
    {
        **F1_RECORDS[case_id],
        "case_id": case_id,
    }
    for case_id in sorted(test_case_ids)
])

assert len(fine_tuned_df) == 200
assert len(frozen_f1_test_df) == 200

for frame in [
    fine_tuned_df,
    frozen_f1_test_df,
]:
    frame["gold_label"] = (
        frame["gold_binary_label"]
        .astype(str)
        .str.upper()
    )

    frame["prediction"] = (
        frame["prediction"]
        .astype(str)
        .str.upper()
    )

    frame["valid_prediction"] = (
        frame["prediction"].isin(LABELS)
    )

    frame["correct"] = (
        frame["valid_prediction"]
        &
        (
            frame["prediction"]
            == frame["gold_label"]
        )
    )

def summarize_model(frame, model_name):
    valid = frame[
        frame["valid_prediction"]
    ].copy()

    summary = {
        "model": model_name,
        "valid_predictions": int(len(valid)),
        "exact_schema_rate": float(
            frame["schema_exact"].mean()
        ),
        "accuracy": float(
            accuracy_score(
                valid["gold_label"],
                valid["prediction"],
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                valid["gold_label"],
                valid["prediction"],
            )
        ),
        "macro_f1": float(
            f1_score(
                valid["gold_label"],
                valid["prediction"],
                average="macro",
            )
        ),
    }

    for family in [
        "normal",
        "lag",
        "wrong_partner",
        "silent_partner",
    ]:
        family_frame = valid[
            valid["case_family"] == family
        ]

        summary[
            f"{family}_accuracy"
        ] = float(
            family_frame["correct"].mean()
        )

    for family in [
        "normal",
        "lag",
        "wrong_partner",
        "silent_partner",
    ]:
        family_frame = valid[
            valid["case_family"] == family
        ]

        semantic_counts = (
            family_frame[
                "semantic_assessment"
            ].value_counts()
        )

        family_total = max(
            len(family_frame),
            1,
        )

        for semantic_value in [
            "COMPATIBLE",
            "INCOMPATIBLE",
            "LIMITED",
        ]:
            summary[
                f"{family}_{semantic_value}_percent"
            ] = (
                100.0
                * int(
                    semantic_counts.get(
                        semantic_value,
                        0,
                    )
                )
                / family_total
            )

    return summary

comparison_summary_df = pd.DataFrame([
    summarize_model(
        frozen_f1_test_df,
        "Frozen F1",
    ),
    summarize_model(
        fine_tuned_df,
        "Fine-tuned F1",
    ),
])

print("=" * 100)
print("HELD-OUT COMPARISON SUMMARY")
print("=" * 100)

display(comparison_summary_df)

def display_confusion(frame, title):
    valid = frame[
        frame["valid_prediction"]
    ]

    matrix = confusion_matrix(
        valid["gold_label"],
        valid["prediction"],
        labels=LABELS,
    )

    confusion_df = pd.DataFrame(
        matrix,
        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],
        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )

    print("\n" + title)
    display(confusion_df)

display_confusion(
    frozen_f1_test_df,
    "FROZEN F1 CONFUSION MATRIX",
)

display_confusion(
    fine_tuned_df,
    "FINE-TUNED F1 CONFUSION MATRIX",
)

paired = (
    frozen_f1_test_df[
        [
            "case_id",
            "correct",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    ]
    .rename(columns={
        column: f"frozen_{column}"
        for column in [
            "correct",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    })
    .merge(
        fine_tuned_df[
            [
                "case_id",
                "case_family",
                "gold_label",
                "correct",
                "prediction",
                "participation_assessment",
                "local_temporal_assessment",
                "global_temporal_assessment",
                "temporal_assessment",
                "semantic_assessment",
                "decisive_dimension",
            ]
        ].rename(columns={
            column: f"fine_tuned_{column}"
            for column in [
                "correct",
                "prediction",
                "participation_assessment",
                "local_temporal_assessment",
                "global_temporal_assessment",
                "temporal_assessment",
                "semantic_assessment",
                "decisive_dimension",
            ]
        }),
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(paired) == 200

branch_stability_rows = []

for field in [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "decisive_dimension",
    "prediction",
]:
    same = (
        paired[f"frozen_{field}"]
        == paired[f"fine_tuned_{field}"]
    )

    branch_stability_rows.append({
        "field": field,
        "same_count": int(same.sum()),
        "same_percent": float(
            100.0 * same.mean()
        ),
        "changed_count": int((~same).sum()),
    })

branch_stability_df = pd.DataFrame(
    branch_stability_rows
)

print("\n" + "=" * 100)
print("NON-SEMANTIC BRANCH PRESERVATION")
print("=" * 100)

display(branch_stability_df)

b = int(
    (
        paired["frozen_correct"]
        &
        ~paired["fine_tuned_correct"]
    ).sum()
)

c = int(
    (
        ~paired["frozen_correct"]
        &
        paired["fine_tuned_correct"]
    ).sum()
)

mcnemar_p = (
    float(
        binomtest(
            min(b, c),
            n=b + c,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )
    if b + c > 0
    else 1.0
)

print("\nPaired prediction changes")
print("Frozen correct / Fine-tuned wrong (b):", b)
print("Frozen wrong / Fine-tuned correct (c):", c)
print("Exact McNemar p-value:", mcnemar_p)

def reasoning_inconsistency_flags(frame):
    frame = frame.copy()

    frame["normal_with_invalid_participation"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["participation_assessment"]
            == "INVALID"
        )
    )

    frame["normal_with_anomalous_temporal"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["temporal_assessment"]
            == "ANOMALOUS"
        )
    )

    frame["normal_with_incompatible_semantics"] = (
        (frame["prediction"] == "NORMAL")
        &
        (
            frame["semantic_assessment"]
            == "INCOMPATIBLE"
        )
    )

    frame["anomalous_without_explicit_failure"] = (
        (frame["prediction"] == "ANOMALOUS")
        &
        (
            frame["participation_assessment"]
            != "INVALID"
        )
        &
        (
            frame["temporal_assessment"]
            != "ANOMALOUS"
        )
        &
        (
            frame["semantic_assessment"]
            != "INCOMPATIBLE"
        )
    )

    return frame

frozen_consistency_df = reasoning_inconsistency_flags(
    frozen_f1_test_df
)

fine_tuned_consistency_df = reasoning_inconsistency_flags(
    fine_tuned_df
)

consistency_summary_df = pd.DataFrame([
    {
        "model": "Frozen F1",
        **{
            column: int(
                frozen_consistency_df[column].sum()
            )
            for column in [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
                "anomalous_without_explicit_failure",
            ]
        },
    },
    {
        "model": "Fine-tuned F1",
        **{
            column: int(
                fine_tuned_consistency_df[column].sum()
            )
            for column in [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
                "anomalous_without_explicit_failure",
            ]
        },
    },
])

print("\n" + "=" * 100)
print("LOGICAL CONSISTENCY")
print("=" * 100)

display(consistency_summary_df)

comparison_summary_df.to_csv(
    TRAINING_RUN_DIR
    / "held_out_comparison_summary.csv",
    index=False,
)

branch_stability_df.to_csv(
    TRAINING_RUN_DIR
    / "branch_stability.csv",
    index=False,
)

consistency_summary_df.to_csv(
    TRAINING_RUN_DIR
    / "logical_consistency_summary.csv",
    index=False,
)

paired.to_csv(
    TRAINING_RUN_DIR
    / "paired_case_level_comparison.csv",
    index=False,
)

fine_tuned_df.to_csv(
    TRAINING_RUN_DIR
    / "fine_tuned_held_out_predictions.csv",
    index=False,
)

print("\nSaved final results to:", TRAINING_RUN_DIR)


HELD-OUT COMPARISON SUMMARY


,model,valid_predictions,exact_schema_rate,accuracy,balanced_accuracy,macro_f1,normal_accuracy,lag_accuracy,wrong_partner_accuracy,silent_partner_accuracy,...,normal_LIMITED_percent,lag_COMPATIBLE_percent,lag_INCOMPATIBLE_percent,lag_LIMITED_percent,wrong_partner_COMPATIBLE_percent,wrong_partner_INCOMPATIBLE_percent,wrong_partner_LIMITED_percent,silent_partner_COMPATIBLE_percent,silent_partner_INCOMPATIBLE_percent,silent_partner_LIMITED_percent
0,Frozen F1,200,1.0,0.885,0.843333,0.845632,0.76,0.86,0.92,1.0,...,6.0,92.0,0.0,8.0,38.0,38.0,24.0,12.0,68.0,20.0
1,Fine-tuned F1,200,1.0,0.875,0.870000,0.844135,0.86,0.78,0.86,1.0,...,4.0,92.0,2.0,6.0,24.0,66.0,10.0,8.0,44.0,48.0



FROZEN F1 CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,38,12
Gold ANOMALOUS,11,139



FINE-TUNED F1 CONFUSION MATRIX


...


NON-SEMANTIC BRANCH PRESERVATION


,field,same_count,same_percent,changed_count
0,participation_assessment,200,100.0,0
1,local_temporal_assessment,174,87.0,26
2,global_temporal_assessment,169,84.5,31
3,temporal_assessment,169,84.5,31
4,decisive_dimension,135,67.5,65
5,prediction,178,89.0,22



Paired prediction changes
Frozen correct / Fine-tuned wrong (b): 12
Frozen wrong / Fine-tuned correct (c): 10
Exact McNemar p-value: 0.8318119049072266

LOGICAL CONSISTENCY


,model,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,anomalous_without_explicit_failure
0,Frozen F1,0,0,0,6
1,Fine-tuned F1,0,0,0,2



Saved final results to: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42


## 25. Apply the predefined success criteria

The experiment succeeds only when wrong-partner semantic
`INCOMPATIBLE` increases without a material loss in:

- normal and lag semantic compatibility;
- normal, lag, wrong-partner, and silent-partner accuracy;
- exact JSON schema validity.


In [ ]:

# ============================================================
# SUCCESS-CRITERIA DASHBOARD
#
# This cell does not declare success automatically from accuracy.
# It reports whether the targeted semantic improvement was achieved
# without materially damaging the other branches.
# ============================================================

frozen_summary = (
    comparison_summary_df[
        comparison_summary_df["model"]
        == "Frozen F1"
    ].iloc[0]
)

fine_tuned_summary = (
    comparison_summary_df[
        comparison_summary_df["model"]
        == "Fine-tuned F1"
    ].iloc[0]
)

MAX_ALLOWED_FAMILY_ACCURACY_DROP = 0.05
MAX_ALLOWED_COMPATIBLE_DROP_PERCENT = 5.0

success_checks = {
    "wrong_partner_INCOMPATIBLE_increased": (
        fine_tuned_summary[
            "wrong_partner_INCOMPATIBLE_percent"
        ]
        >
        frozen_summary[
            "wrong_partner_INCOMPATIBLE_percent"
        ]
    ),

    "normal_COMPATIBLE_preserved": (
        fine_tuned_summary[
            "normal_COMPATIBLE_percent"
        ]
        >=
        frozen_summary[
            "normal_COMPATIBLE_percent"
        ]
        - MAX_ALLOWED_COMPATIBLE_DROP_PERCENT
    ),

    "lag_COMPATIBLE_preserved": (
        fine_tuned_summary[
            "lag_COMPATIBLE_percent"
        ]
        >=
        frozen_summary[
            "lag_COMPATIBLE_percent"
        ]
        - MAX_ALLOWED_COMPATIBLE_DROP_PERCENT
    ),

    "normal_accuracy_preserved": (
        fine_tuned_summary[
            "normal_accuracy"
        ]
        >=
        frozen_summary[
            "normal_accuracy"
        ]
        - MAX_ALLOWED_FAMILY_ACCURACY_DROP
    ),

    "lag_accuracy_preserved": (
        fine_tuned_summary[
            "lag_accuracy"
        ]
        >=
        frozen_summary[
            "lag_accuracy"
        ]
        - MAX_ALLOWED_FAMILY_ACCURACY_DROP
    ),

    "wrong_partner_accuracy_preserved": (
        fine_tuned_summary[
            "wrong_partner_accuracy"
        ]
        >=
        frozen_summary[
            "wrong_partner_accuracy"
        ]
        - MAX_ALLOWED_FAMILY_ACCURACY_DROP
    ),

    "silent_partner_accuracy_preserved": (
        fine_tuned_summary[
            "silent_partner_accuracy"
        ]
        >=
        frozen_summary[
            "silent_partner_accuracy"
        ]
        - MAX_ALLOWED_FAMILY_ACCURACY_DROP
    ),

    "exact_schema_rate_100_percent": (
        fine_tuned_summary[
            "exact_schema_rate"
        ]
        == 1.0
    ),
}

success_checks_df = pd.DataFrame([
    {
        "criterion": criterion,
        "passed": bool(passed),
    }
    for criterion, passed in success_checks.items()
])

display(success_checks_df)

all_checks_passed = all(
    success_checks.values()
)

print(
    "All predefined checks passed:",
    all_checks_passed,
)

SUCCESS_REPORT_PATH = (
    TRAINING_RUN_DIR
    / "success_criteria.json"
)

SUCCESS_REPORT_PATH.write_text(
    json.dumps(
        {
            "all_checks_passed": all_checks_passed,
            "checks": {
                key: bool(value)
                for key, value
                in success_checks.items()
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Success report:", SUCCESS_REPORT_PATH)


,criterion,passed
0,wrong_partner_INCOMPATIBLE_increased,True
1,normal_COMPATIBLE_preserved,True
2,lag_COMPATIBLE_preserved,True
3,normal_accuracy_preserved,True
4,lag_accuracy_preserved,False
5,wrong_partner_accuracy_preserved,False
6,silent_partner_accuracy_preserved,True
7,exact_schema_rate_100_percent,True


All predefined checks passed: False
Success report: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/success_criteria.json


## 26. Optional Colab runtime disconnection

Run only after all adapter, cache, and evaluation files have been saved.


In [ ]:
from google.colab import runtime

runtime.unassign()


In [ ]:
# ============================================================
# POST-HOC DETAILED ANALYSIS
# LOAD SAVED FROZEN F1 + FINE-TUNED F1 RESULTS
#
# No model loading.
# No GPU required.
# No inference.
# No training.
# ============================================================

from google.colab import drive
drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path
import json
import pandas as pd
from IPython.display import display


# ============================================================
# 1. EXACT SAVED PATHS
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)

FROZEN_F1_CACHE_PATH = (
    OUT_DIR
    / "structured_r1_improved_semantic_payload_ablation"
    / "f1"
    / "predictions_cache.json"
)

TRAINING_RUN_DIR = (
    OUT_DIR
    / "structured_r1_f1_semantic_targeted_finetuning"
    / "qlora_r8_lambda_0_5_seed_42"
)

FINE_TUNED_CACHE_PATH = (
    TRAINING_RUN_DIR
    / "held_out_test_predictions_cache.json"
)

POSTHOC_OUTPUT_DIR = (
    TRAINING_RUN_DIR
    / "posthoc_detailed_analysis"
)

POSTHOC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. VERIFY FILES
# ============================================================

assert FROZEN_F1_CACHE_PATH.exists(), (
    "Frozen F1 cache not found:\n"
    f"{FROZEN_F1_CACHE_PATH}"
)

assert FINE_TUNED_CACHE_PATH.exists(), (
    "Fine-tuned test cache not found:\n"
    f"{FINE_TUNED_CACHE_PATH}"
)

print("=" * 100)
print("RESULT FILES")
print("=" * 100)
print("Frozen F1:", FROZEN_F1_CACHE_PATH)
print("Fine-tuned F1:", FINE_TUNED_CACHE_PATH)


# ============================================================
# 3. LOAD JSON CACHES
# ============================================================

with FROZEN_F1_CACHE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    frozen_cache = json.load(file)

with FINE_TUNED_CACHE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    fine_tuned_cache = json.load(file)

assert isinstance(
    frozen_cache.get("records"),
    dict,
)

assert isinstance(
    fine_tuned_cache.get("records"),
    dict,
)

print("\nFrozen cache records:", len(
    frozen_cache["records"]
))

print("Fine-tuned cache records:", len(
    fine_tuned_cache["records"]
))


# ============================================================
# 4. CONVERT RECORDS TO DATAFRAMES
# ============================================================

def records_to_dataframe(cache):
    rows = []

    for case_id, record in cache["records"].items():
        row = dict(record)
        row["case_id"] = str(
            row.get("case_id", case_id)
        )
        rows.append(row)

    return pd.DataFrame(rows)


frozen_all_df = records_to_dataframe(
    frozen_cache
)

fine_tuned_df = records_to_dataframe(
    fine_tuned_cache
)

assert len(fine_tuned_df) == 200, (
    "Expected exactly 200 fine-tuned held-out cases, "
    f"found {len(fine_tuned_df)}."
)


# ============================================================
# 5. SELECT THE SAME 200 CASES FROM FROZEN F1
# ============================================================

test_case_ids = set(
    fine_tuned_df["case_id"].astype(str)
)

frozen_f1_test_df = (
    frozen_all_df[
        frozen_all_df[
            "case_id"
        ].astype(str).isin(test_case_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(frozen_f1_test_df) == 200, (
    "Could not recover all 200 test cases "
    "from the frozen F1 cache."
)

assert set(
    frozen_f1_test_df["case_id"].astype(str)
) == test_case_ids


# ============================================================
# 6. NORMALISE IMPORTANT COLUMNS
# ============================================================

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]

SEMANTIC_VALUES = [
    "COMPATIBLE",
    "INCOMPATIBLE",
    "LIMITED",
]

FAMILY_ORDER = [
    "normal",
    "lag",
    "wrong_partner",
    "silent_partner",
]


def normalise_result_frame(frame):
    frame = frame.copy()

    frame["case_id"] = (
        frame["case_id"]
        .astype(str)
    )

    frame["case_family"] = (
        frame["case_family"]
        .astype(str)
        .str.lower()
    )

    if "gold_binary_label" in frame.columns:
        frame["gold_label"] = (
            frame["gold_binary_label"]
            .astype(str)
            .str.upper()
        )
    else:
        frame["gold_label"] = (
            frame["gold_label"]
            .astype(str)
            .str.upper()
        )

    frame["prediction"] = (
        frame["prediction"]
        .astype(str)
        .str.upper()
    )

    frame["semantic_assessment"] = (
        frame["semantic_assessment"]
        .astype(str)
        .str.upper()
    )

    for column in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "decisive_dimension",
    ]:
        frame[column] = (
            frame[column]
            .astype(str)
            .str.upper()
        )

    frame["valid_prediction"] = (
        frame["prediction"].isin(LABELS)
    )

    frame["correct"] = (
        frame["valid_prediction"]
        &
        (
            frame["prediction"]
            == frame["gold_label"]
        )
    )

    return frame


frozen_f1_test_df = normalise_result_frame(
    frozen_f1_test_df
)

fine_tuned_df = normalise_result_frame(
    fine_tuned_df
)


# ============================================================
# 7. DATASET AUDIT
# ============================================================

print("\n" + "=" * 100)
print("TEST-SET FAMILY AUDIT")
print("=" * 100)

family_audit = (
    fine_tuned_df[
        "case_family"
    ]
    .value_counts()
    .reindex(
        FAMILY_ORDER,
        fill_value=0,
    )
    .rename("cases")
    .to_frame()
)

display(family_audit)

assert family_audit["cases"].to_dict() == {
    "normal": 50,
    "lag": 50,
    "wrong_partner": 50,
    "silent_partner": 50,
}


# ============================================================
# 8. SEMANTIC COUNTS FUNCTION
# ============================================================

def semantic_counts_table(frame):
    table = pd.crosstab(
        index=frame["case_family"],
        columns=frame["semantic_assessment"],
    )

    table = table.reindex(
        index=FAMILY_ORDER,
        columns=SEMANTIC_VALUES,
        fill_value=0,
    )

    table["All"] = table.sum(axis=1)

    total_row = pd.DataFrame(
        [table.sum(axis=0)],
        index=["All"],
    )

    return pd.concat(
        [
            table,
            total_row,
        ]
    )


def semantic_percentages_table(frame):
    counts = pd.crosstab(
        index=frame["case_family"],
        columns=frame["semantic_assessment"],
    )

    counts = counts.reindex(
        index=FAMILY_ORDER,
        columns=SEMANTIC_VALUES,
        fill_value=0,
    )

    percentages = (
        counts.div(
            counts.sum(axis=1),
            axis=0,
        )
        * 100.0
    ).round(2)

    return percentages


# ============================================================
# 9. FROZEN F1 SEMANTIC DISTRIBUTION
# ============================================================

frozen_semantic_counts = semantic_counts_table(
    frozen_f1_test_df
)

frozen_semantic_percentages = (
    semantic_percentages_table(
        frozen_f1_test_df
    )
)

print("\n" + "=" * 100)
print("FROZEN F1 — SEMANTIC ASSESSMENT COUNTS")
print("=" * 100)

display(frozen_semantic_counts)

print("\nFROZEN F1 — SEMANTIC ASSESSMENT PERCENTAGES")

display(frozen_semantic_percentages)


# ============================================================
# 10. FINE-TUNED F1 SEMANTIC DISTRIBUTION
# ============================================================

fine_tuned_semantic_counts = semantic_counts_table(
    fine_tuned_df
)

fine_tuned_semantic_percentages = (
    semantic_percentages_table(
        fine_tuned_df
    )
)

print("\n" + "=" * 100)
print("FINE-TUNED F1 — SEMANTIC ASSESSMENT COUNTS")
print("=" * 100)

display(fine_tuned_semantic_counts)

print("\nFINE-TUNED F1 — SEMANTIC ASSESSMENT PERCENTAGES")

display(fine_tuned_semantic_percentages)


# ============================================================
# 11. SIDE-BY-SIDE SEMANTIC COUNTS
# ============================================================

semantic_side_by_side = pd.concat(
    {
        "Frozen F1": (
            frozen_semantic_counts.loc[
                FAMILY_ORDER
            ]
        ),
        "Fine-tuned F1": (
            fine_tuned_semantic_counts.loc[
                FAMILY_ORDER
            ]
        ),
    },
    axis=1,
)

print("\n" + "=" * 100)
print("SEMANTIC COUNTS — FROZEN vs FINE-TUNED")
print("=" * 100)

display(semantic_side_by_side)


# ============================================================
# 12. BUILD CASE-LEVEL PAIRED TABLE
# ============================================================

comparison_fields = [
    "prediction",
    "correct",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]

frozen_for_merge = (
    frozen_f1_test_df[
        [
            "case_id",
            *comparison_fields,
        ]
    ]
    .rename(
        columns={
            column: f"frozen_{column}"
            for column in comparison_fields
        }
    )
)

fine_tuned_for_merge = (
    fine_tuned_df[
        [
            "case_id",
            "source_group_id",
            "case_family",
            "case_variant",
            "gold_label",
            *comparison_fields,
        ]
    ]
    .rename(
        columns={
            column: f"fine_tuned_{column}"
            for column in comparison_fields
        }
    )
)

paired_detailed_df = (
    fine_tuned_for_merge
    .merge(
        frozen_for_merge,
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(paired_detailed_df) == 200


# ============================================================
# 13. ADD CHANGE / TRANSITION COLUMNS
# ============================================================

paired_detailed_df[
    "semantic_transition"
] = (
    paired_detailed_df[
        "frozen_semantic_assessment"
    ]
    + " → "
    + paired_detailed_df[
        "fine_tuned_semantic_assessment"
    ]
)

paired_detailed_df[
    "semantic_changed"
] = (
    paired_detailed_df[
        "frozen_semantic_assessment"
    ]
    !=
    paired_detailed_df[
        "fine_tuned_semantic_assessment"
    ]
)

paired_detailed_df[
    "prediction_transition"
] = (
    paired_detailed_df[
        "frozen_prediction"
    ]
    + " → "
    + paired_detailed_df[
        "fine_tuned_prediction"
    ]
)

paired_detailed_df[
    "prediction_changed"
] = (
    paired_detailed_df[
        "frozen_prediction"
    ]
    !=
    paired_detailed_df[
        "fine_tuned_prediction"
    ]
)


def outcome_change(row):
    frozen_correct = bool(
        row["frozen_correct"]
    )

    fine_correct = bool(
        row["fine_tuned_correct"]
    )

    if frozen_correct and fine_correct:
        return "BOTH_CORRECT"

    if frozen_correct and not fine_correct:
        return "REGRESSION"

    if not frozen_correct and fine_correct:
        return "IMPROVEMENT"

    return "BOTH_WRONG"


paired_detailed_df[
    "classification_outcome"
] = paired_detailed_df.apply(
    outcome_change,
    axis=1,
)


# ============================================================
# 14. SEMANTIC TRANSITIONS — ALL CASES
# ============================================================

print("\n" + "=" * 100)
print("SEMANTIC TRANSITIONS — ALL 200 CASES")
print("=" * 100)

all_semantic_transitions = pd.crosstab(
    index=paired_detailed_df[
        "frozen_semantic_assessment"
    ],
    columns=paired_detailed_df[
        "fine_tuned_semantic_assessment"
    ],
    margins=True,
)

all_semantic_transitions = (
    all_semantic_transitions.reindex(
        index=[
            *SEMANTIC_VALUES,
            "All",
        ],
        columns=[
            *SEMANTIC_VALUES,
            "All",
        ],
        fill_value=0,
    )
)

display(all_semantic_transitions)


# ============================================================
# 15. SEMANTIC TRANSITIONS PER CASE FAMILY
# ============================================================

for family in FAMILY_ORDER:
    family_df = paired_detailed_df[
        paired_detailed_df[
            "case_family"
        ] == family
    ]

    transition_table = pd.crosstab(
        index=family_df[
            "frozen_semantic_assessment"
        ],
        columns=family_df[
            "fine_tuned_semantic_assessment"
        ],
        margins=True,
    )

    transition_table = transition_table.reindex(
        index=[
            *SEMANTIC_VALUES,
            "All",
        ],
        columns=[
            *SEMANTIC_VALUES,
            "All",
        ],
        fill_value=0,
    )

    print("\n" + "=" * 100)
    print(
        f"{family.upper()} — "
        "FROZEN SEMANTIC → FINE-TUNED SEMANTIC"
    )
    print("=" * 100)

    display(transition_table)


# ============================================================
# 16. CLASSIFICATION OUTCOMES PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("CLASSIFICATION OUTCOME CHANGES PER FAMILY")
print("=" * 100)

classification_changes_table = pd.crosstab(
    index=paired_detailed_df[
        "case_family"
    ],
    columns=paired_detailed_df[
        "classification_outcome"
    ],
    margins=True,
)

display(classification_changes_table)


# ============================================================
# 17. ONLY CASES WHOSE SEMANTIC ASSESSMENT CHANGED
# ============================================================

semantic_changed_cases_df = (
    paired_detailed_df[
        paired_detailed_df[
            "semantic_changed"
        ]
    ]
    .copy()
    .sort_values(
        [
            "case_family",
            "semantic_transition",
            "case_id",
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("CASES WITH CHANGED SEMANTIC ASSESSMENT")
print("=" * 100)

print(
    "Changed semantic assessments:",
    len(semantic_changed_cases_df),
)

display(
    semantic_changed_cases_df[
        [
            "case_id",
            "source_group_id",
            "case_family",
            "gold_label",
            "frozen_semantic_assessment",
            "fine_tuned_semantic_assessment",
            "semantic_transition",
            "frozen_prediction",
            "fine_tuned_prediction",
            "prediction_transition",
            "frozen_correct",
            "fine_tuned_correct",
            "classification_outcome",
        ]
    ]
)


# ============================================================
# 18. WRONG-PARTNER CASES — DETAILED
# ============================================================

wrong_partner_detailed_df = (
    paired_detailed_df[
        paired_detailed_df[
            "case_family"
        ] == "wrong_partner"
    ]
    .copy()
    .sort_values(
        [
            "semantic_transition",
            "case_id",
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("ALL 50 WRONG-PARTNER CASES — DETAILED")
print("=" * 100)

display(
    wrong_partner_detailed_df[
        [
            "case_id",
            "source_group_id",
            "gold_label",
            "frozen_semantic_assessment",
            "fine_tuned_semantic_assessment",
            "semantic_transition",
            "frozen_temporal_assessment",
            "fine_tuned_temporal_assessment",
            "frozen_decisive_dimension",
            "fine_tuned_decisive_dimension",
            "frozen_prediction",
            "fine_tuned_prediction",
            "frozen_correct",
            "fine_tuned_correct",
            "classification_outcome",
        ]
    ]
)


# ============================================================
# 19. EVERY CASE, SEPARATED BY FAMILY
# ============================================================

columns_to_display = [
    "case_id",
    "source_group_id",
    "case_family",
    "gold_label",

    "frozen_participation_assessment",
    "fine_tuned_participation_assessment",

    "frozen_local_temporal_assessment",
    "fine_tuned_local_temporal_assessment",

    "frozen_global_temporal_assessment",
    "fine_tuned_global_temporal_assessment",

    "frozen_temporal_assessment",
    "fine_tuned_temporal_assessment",

    "frozen_semantic_assessment",
    "fine_tuned_semantic_assessment",
    "semantic_transition",

    "frozen_decisive_dimension",
    "fine_tuned_decisive_dimension",

    "frozen_prediction",
    "fine_tuned_prediction",
    "prediction_transition",

    "frozen_correct",
    "fine_tuned_correct",
    "classification_outcome",
]

for family in FAMILY_ORDER:
    print("\n" + "=" * 100)
    print(
        f"DETAILED CASE-LEVEL RESULTS — "
        f"{family.upper()}"
    )
    print("=" * 100)

    family_detailed_df = (
        paired_detailed_df[
            paired_detailed_df[
                "case_family"
            ] == family
        ]
        .sort_values("case_id")
        .reset_index(drop=True)
    )

    display(
        family_detailed_df[
            columns_to_display
        ]
    )


# ============================================================
# 20. SAVE ALL POST-HOC TABLES
# ============================================================

frozen_semantic_counts.to_csv(
    POSTHOC_OUTPUT_DIR
    / "frozen_semantic_counts_by_family.csv"
)

fine_tuned_semantic_counts.to_csv(
    POSTHOC_OUTPUT_DIR
    / "fine_tuned_semantic_counts_by_family.csv"
)

frozen_semantic_percentages.to_csv(
    POSTHOC_OUTPUT_DIR
    / "frozen_semantic_percentages_by_family.csv"
)

fine_tuned_semantic_percentages.to_csv(
    POSTHOC_OUTPUT_DIR
    / "fine_tuned_semantic_percentages_by_family.csv"
)

all_semantic_transitions.to_csv(
    POSTHOC_OUTPUT_DIR
    / "semantic_transition_matrix_all_cases.csv"
)

classification_changes_table.to_csv(
    POSTHOC_OUTPUT_DIR
    / "classification_outcome_changes.csv"
)

semantic_changed_cases_df.to_csv(
    POSTHOC_OUTPUT_DIR
    / "cases_with_changed_semantic_assessment.csv",
    index=False,
)

wrong_partner_detailed_df.to_csv(
    POSTHOC_OUTPUT_DIR
    / "wrong_partner_detailed_comparison.csv",
    index=False,
)

paired_detailed_df.to_csv(
    POSTHOC_OUTPUT_DIR
    / "all_200_cases_detailed_comparison.csv",
    index=False,
)

print("\n" + "=" * 100)
print("POST-HOC ANALYSIS COMPLETE")
print("=" * 100)

print("Saved results to:")
print(POSTHOC_OUTPUT_DIR)

Mounted at /content/drive
RESULT FILES
Frozen F1: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_improved_semantic_payload_ablation/f1/predictions_cache.json
Fine-tuned F1: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/held_out_test_predictions_cache.json

Frozen cache records: 400
Fine-tuned cache records: 200

TEST-SET FAMILY AUDIT


,cases
case_family,
normal,50
lag,50
wrong_partner,50
silent_partner,50



FROZEN F1 — SEMANTIC ASSESSMENT COUNTS


semantic_assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
normal,47,0,3,50
lag,46,0,4,50
wrong_partner,19,19,12,50
silent_partner,6,34,10,50
All,118,53,29,200



FROZEN F1 — SEMANTIC ASSESSMENT PERCENTAGES


...


DETAILED CASE-LEVEL RESULTS — WRONG_PARTNER


,case_id,source_group_id,case_family,gold_label,frozen_participation_assessment,fine_tuned_participation_assessment,frozen_local_temporal_assessment,fine_tuned_local_temporal_assessment,frozen_global_temporal_assessment,fine_tuned_global_temporal_assessment,...,fine_tuned_semantic_assessment,semantic_transition,frozen_decisive_dimension,fine_tuned_decisive_dimension,frozen_prediction,fine_tuned_prediction,prediction_transition,frozen_correct,fine_tuned_correct,classification_outcome
0,consolidation_wrong_partner_000,heldout_source_000,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,TEMPORAL,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
1,consolidation_wrong_partner_002,heldout_source_002,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,NORMAL,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,TEMPORAL,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
2,consolidation_wrong_partner_004,heldout_source_004,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,TEMPORAL,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
3,consolidation_wrong_partner_006,heldout_source_006,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,COMPATIBLE,COMPATIBLE → COMPATIBLE,TEMPORAL,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
4,consolidation_wrong_partner_007,heldout_source_007,wrong_partner,ANOMALOUS,VALID,VALID,NORMAL,NORMAL,ANOMALOUS,NORMAL,...,COMPATIBLE,LIMITED → COMPATIBLE,TEMPORAL,SEMANTIC,ANOMALOUS,NORMAL,ANOMALOUS → NORMAL,True,False,REGRESSION
5,consolidation_wrong_partner_010,heldout_source_010,wrong_partner,ANOMALOUS,VALID,VALID,NORMAL,NORMAL,NORMAL,NORMAL,...,COMPATIBLE,LIMITED → COMPATIBLE,SEMANTIC,SEMANTIC,ANOMALOUS,NORMAL,ANOMALOUS → NORMAL,True,False,REGRESSION
6,consolidation_wrong_partner_018,heldout_source_018,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,COMPATIBLE,COMPATIBLE → COMPATIBLE,TEMPORAL,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
7,consolidation_wrong_partner_019,heldout_source_019,wrong_partner,ANOMALOUS,VALID,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,COMPATIBLE,COMPATIBLE → COMPATIBLE,TEMPORAL,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
8,consolidation_wrong_partner_020,heldout_source_020,wrong_partner,ANOMALOUS,VALID,VALID,NORMAL,NORMAL,ANOMALOUS,NORMAL,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,TEMPORAL,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
9,consolidation_wrong_partner_021,heldout_source_021,wrong_partner,ANOMALOUS,VALID,VALID,NORMAL,ANOMALOUS,NORMAL,ANOMALOUS,...,COMPATIBLE,COMPATIBLE → COMPATIBLE,SEMANTIC,TEMPORAL,NORMAL,ANOMALOUS,NORMAL → ANOMALOUS,False,True,IMPROVEMENT



DETAILED CASE-LEVEL RESULTS — SILENT_PARTNER


,case_id,source_group_id,case_family,gold_label,frozen_participation_assessment,fine_tuned_participation_assessment,frozen_local_temporal_assessment,fine_tuned_local_temporal_assessment,frozen_global_temporal_assessment,fine_tuned_global_temporal_assessment,...,fine_tuned_semantic_assessment,semantic_transition,frozen_decisive_dimension,fine_tuned_decisive_dimension,frozen_prediction,fine_tuned_prediction,prediction_transition,frozen_correct,fine_tuned_correct,classification_outcome
0,consolidation_silent_partner_000,heldout_source_000,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,SEMANTIC,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
1,consolidation_silent_partner_002,heldout_source_002,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,PARTICIPATION,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
2,consolidation_silent_partner_004,heldout_source_004,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,LIMITED,ANOMALOUS,LIMITED,...,LIMITED,INCOMPATIBLE → LIMITED,SEMANTIC,PARTICIPATION,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
3,consolidation_silent_partner_006,heldout_source_006,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,LIMITED,LIMITED → LIMITED,PARTICIPATION,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
4,consolidation_silent_partner_007,heldout_source_007,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,SEMANTIC,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
5,consolidation_silent_partner_010,heldout_source_010,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,SEMANTIC,SEMANTIC,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
6,consolidation_silent_partner_018,heldout_source_018,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,INCOMPATIBLE,INCOMPATIBLE → INCOMPATIBLE,SEMANTIC,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
7,consolidation_silent_partner_019,heldout_source_019,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,LIMITED,ANOMALOUS,LIMITED,...,LIMITED,INCOMPATIBLE → LIMITED,PARTICIPATION,PARTICIPATION,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
8,consolidation_silent_partner_020,heldout_source_020,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,COMPATIBLE,COMPATIBLE → COMPATIBLE,PARTICIPATION,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT
9,consolidation_silent_partner_021,heldout_source_021,silent_partner,ANOMALOUS,INVALID,INVALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,LIMITED,INCOMPATIBLE → LIMITED,PARTICIPATION,TEMPORAL,ANOMALOUS,ANOMALOUS,ANOMALOUS → ANOMALOUS,True,True,BOTH_CORRECT



POST-HOC ANALYSIS COMPLETE
Saved results to:
/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/posthoc_detailed_analysis


In [ ]:
# ============================================================
# FINE-TUNED F1 — CLEAN HELD-OUT TEST ANALYSIS
#
# Loads the saved 200 fine-tuned predictions.
# No model loading.
# No inference.
# No training.
# ============================================================

from google.colab import drive
drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path
import json
import pandas as pd
from IPython.display import display


# ============================================================
# 1. LOAD SAVED FINE-TUNED RESULTS
# ============================================================

TRAINING_RUN_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "structured_r1_f1_semantic_targeted_finetuning/"
    "qlora_r8_lambda_0_5_seed_42"
)

FINE_TUNED_TEST_CACHE_PATH = (
    TRAINING_RUN_DIR
    / "held_out_test_predictions_cache.json"
)

assert FINE_TUNED_TEST_CACHE_PATH.exists(), (
    "Fine-tuned cache not found:\n"
    f"{FINE_TUNED_TEST_CACHE_PATH}"
)

with FINE_TUNED_TEST_CACHE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    fine_tuned_cache = json.load(file)

records = fine_tuned_cache["records"]

fine_tuned_df = pd.DataFrame(
    list(records.values())
)

assert len(fine_tuned_df) == 200, (
    f"Expected 200 cases, found {len(fine_tuned_df)}."
)

print("Loaded fine-tuned predictions:", len(fine_tuned_df))
print("Cache:", FINE_TUNED_TEST_CACHE_PATH)


# ============================================================
# 2. NORMALISE COLUMNS
# ============================================================

fine_tuned_df["case_id"] = (
    fine_tuned_df["case_id"]
    .astype(str)
)

fine_tuned_df["case_family"] = (
    fine_tuned_df["case_family"]
    .astype(str)
    .str.lower()
)

fine_tuned_df["gold_label"] = (
    fine_tuned_df["gold_binary_label"]
    .astype(str)
    .str.upper()
)

fine_tuned_df["prediction"] = (
    fine_tuned_df["prediction"]
    .astype(str)
    .str.upper()
)

for column in [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]:
    fine_tuned_df[column] = (
        fine_tuned_df[column]
        .astype(str)
        .str.upper()
    )

fine_tuned_df["correct"] = (
    fine_tuned_df["prediction"]
    ==
    fine_tuned_df["gold_label"]
)

FAMILY_ORDER = [
    "normal",
    "lag",
    "wrong_partner",
    "silent_partner",
]

SEMANTIC_ORDER = [
    "COMPATIBLE",
    "INCOMPATIBLE",
    "LIMITED",
]


# ============================================================
# 3. NUMBER OF CASES PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — TEST CASES PER FAMILY")
print("=" * 100)

display(
    fine_tuned_df[
        "case_family"
    ]
    .value_counts()
    .reindex(
        FAMILY_ORDER,
        fill_value=0,
    )
    .rename("cases")
    .to_frame()
)


# ============================================================
# 4. FINAL PREDICTIONS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — FINAL PREDICTIONS PER FAMILY")
print("=" * 100)

final_predictions_table = pd.crosstab(
    index=fine_tuned_df["case_family"],
    columns=fine_tuned_df["prediction"],
)

final_predictions_table = (
    final_predictions_table.reindex(
        index=FAMILY_ORDER,
        columns=[
            "NORMAL",
            "ANOMALOUS",
        ],
        fill_value=0,
    )
)

final_predictions_table["All"] = (
    final_predictions_table.sum(axis=1)
)

final_predictions_table = pd.concat(
    [
        final_predictions_table,
        pd.DataFrame(
            [final_predictions_table.sum(axis=0)],
            index=["All"],
        ),
    ]
)

display(final_predictions_table)


# ============================================================
# 5. ACCURACY PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — ACCURACY PER FAMILY")
print("=" * 100)

family_accuracy_df = (
    fine_tuned_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "count",
        ),
        correct_predictions=(
            "correct",
            "sum",
        ),
        accuracy=(
            "correct",
            "mean",
        ),
        predicted_NORMAL=(
            "prediction",
            lambda values: (
                values == "NORMAL"
            ).sum(),
        ),
        predicted_ANOMALOUS=(
            "prediction",
            lambda values: (
                values == "ANOMALOUS"
            ).sum(),
        ),
    )
)

family_accuracy_df["case_family"] = pd.Categorical(
    family_accuracy_df["case_family"],
    categories=FAMILY_ORDER,
    ordered=True,
)

family_accuracy_df = (
    family_accuracy_df
    .sort_values("case_family")
    .reset_index(drop=True)
)

display(family_accuracy_df)


# ============================================================
# 6. SEMANTIC ASSESSMENT PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — SEMANTIC ASSESSMENTS PER FAMILY")
print("=" * 100)

semantic_table = pd.crosstab(
    index=fine_tuned_df["case_family"],
    columns=fine_tuned_df["semantic_assessment"],
)

semantic_table = semantic_table.reindex(
    index=FAMILY_ORDER,
    columns=SEMANTIC_ORDER,
    fill_value=0,
)

semantic_table["All"] = semantic_table.sum(axis=1)

semantic_table = pd.concat(
    [
        semantic_table,
        pd.DataFrame(
            [semantic_table.sum(axis=0)],
            index=["All"],
        ),
    ]
)

display(semantic_table)


# ============================================================
# 7. SEMANTIC PERCENTAGES PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — SEMANTIC PERCENTAGES PER FAMILY")
print("=" * 100)

semantic_percentages = (
    semantic_table
    .loc[FAMILY_ORDER, SEMANTIC_ORDER]
    .div(
        semantic_table.loc[
            FAMILY_ORDER,
            "All",
        ],
        axis=0,
    )
    .mul(100)
    .round(2)
)

display(semantic_percentages)


# ============================================================
# 8. PARTICIPATION ASSESSMENT PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — PARTICIPATION ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=fine_tuned_df["case_family"],
        columns=(
            fine_tuned_df[
                "participation_assessment"
            ]
        ),
        margins=True,
    )
)


# ============================================================
# 9. LOCAL TEMPORAL ASSESSMENT PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — LOCAL TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=fine_tuned_df["case_family"],
        columns=(
            fine_tuned_df[
                "local_temporal_assessment"
            ]
        ),
        margins=True,
    )
)


# ============================================================
# 10. GLOBAL TEMPORAL ASSESSMENT PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — GLOBAL TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=fine_tuned_df["case_family"],
        columns=(
            fine_tuned_df[
                "global_temporal_assessment"
            ]
        ),
        margins=True,
    )
)


# ============================================================
# 11. COMBINED TEMPORAL ASSESSMENT PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=fine_tuned_df["case_family"],
        columns=(
            fine_tuned_df[
                "temporal_assessment"
            ]
        ),
        margins=True,
    )
)


# ============================================================
# 12. DECISIVE DIMENSION PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNED F1 — DECISIVE DIMENSION")
print("=" * 100)

display(
    pd.crosstab(
        index=fine_tuned_df["case_family"],
        columns=(
            fine_tuned_df[
                "decisive_dimension"
            ]
        ),
        margins=True,
    )
)


# ============================================================
# 13. DETAILED RESULTS PER FAMILY
# ============================================================

DETAILED_COLUMNS = [
    "case_id",
    "source_group_id",
    "case_variant",
    "gold_label",
    "prediction",
    "correct",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "schema_exact",
]

for family in FAMILY_ORDER:
    family_df = (
        fine_tuned_df[
            fine_tuned_df[
                "case_family"
            ] == family
        ]
        .copy()
        .sort_values("case_id")
        .reset_index(drop=True)
    )

    print("\n" + "=" * 100)
    print(
        f"FINE-TUNED F1 — "
        f"{family.upper()} CASES"
    )
    print("=" * 100)

    display(
        family_df[
            DETAILED_COLUMNS
        ]
    )


# ============================================================
# 14. WRONG-PARTNER CASES GROUPED BY SEMANTIC RESULT
# ============================================================

wrong_partner_df = (
    fine_tuned_df[
        fine_tuned_df[
            "case_family"
        ] == "wrong_partner"
    ]
    .copy()
)

for semantic_value in SEMANTIC_ORDER:
    subset = (
        wrong_partner_df[
            wrong_partner_df[
                "semantic_assessment"
            ] == semantic_value
        ]
        .sort_values("case_id")
        .reset_index(drop=True)
    )

    print("\n" + "=" * 100)
    print(
        "FINE-TUNED WRONG PARTNER — "
        f"{semantic_value}"
    )
    print("=" * 100)

    print("Cases:", len(subset))

    display(
        subset[
            DETAILED_COLUMNS
        ]
    )


# ============================================================
# 15. SAVE CLEAN TABLE
# ============================================================

OUTPUT_CSV_PATH = (
    TRAINING_RUN_DIR
    / "fine_tuned_clean_case_level_results.csv"
)

fine_tuned_df[
    [
        "case_id",
        "source_group_id",
        "case_family",
        *DETAILED_COLUMNS[2:],
    ]
].to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)

print("\n" + "=" * 100)
print("COMPLETE")
print("=" * 100)

print("Saved clean fine-tuned results to:")
print(OUTPUT_CSV_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded fine-tuned predictions: 200
Cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/held_out_test_predictions_cache.json

FINE-TUNED F1 — TEST CASES PER FAMILY


,cases
case_family,
normal,50
lag,50
wrong_partner,50
silent_partner,50



FINE-TUNED F1 — FINAL PREDICTIONS PER FAMILY


prediction,NORMAL,ANOMALOUS,All
normal,43,7,50
lag,11,39,50
wrong_partner,7,43,50
silent_partner,0,50,50
All,61,139,200



FINE-TUNED F1 — ACCURACY PER FAMILY


...


FINE-TUNED WRONG PARTNER — INCOMPATIBLE
Cases: 33


,case_id,source_group_id,case_variant,gold_label,prediction,correct,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact
0,consolidation_wrong_partner_000,heldout_source_000,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True
1,consolidation_wrong_partner_002,heldout_source_002,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
2,consolidation_wrong_partner_004,heldout_source_004,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
3,consolidation_wrong_partner_020,heldout_source_020,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
4,consolidation_wrong_partner_024,heldout_source_024,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
5,consolidation_wrong_partner_026,heldout_source_026,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
6,consolidation_wrong_partner_027,heldout_source_027,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
7,consolidation_wrong_partner_028,heldout_source_028,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,SEMANTIC,True
8,consolidation_wrong_partner_029,heldout_source_029,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
9,consolidation_wrong_partner_032,heldout_source_032,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True



FINE-TUNED WRONG PARTNER — LIMITED
Cases: 5


,case_id,source_group_id,case_variant,gold_label,prediction,correct,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact
0,consolidation_wrong_partner_049,heldout_source_049,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
1,consolidation_wrong_partner_055,heldout_source_055,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
2,consolidation_wrong_partner_074,heldout_source_074,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
3,consolidation_wrong_partner_078,heldout_source_078,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
4,consolidation_wrong_partner_081,heldout_source_081,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True



COMPLETE
Saved clean fine-tuned results to:
/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/fine_tuned_clean_case_level_results.csv


In [ ]:
# ============================================================
# FROZEN F1 — CLEAN HELD-OUT TEST ANALYSIS
#
# Loads the saved original F1 predictions for the exact same
# 200 cases used in the fine-tuned evaluation.
#
# No model loading.
# No inference.
# No training.
# No GPU required.
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path
import json
import pandas as pd
from IPython.display import display


# ============================================================
# 1. SAVED RESULT PATHS
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)

FROZEN_F1_CACHE_PATH = (
    OUT_DIR
    / "structured_r1_improved_semantic_payload_ablation"
    / "f1"
    / "predictions_cache.json"
)

TRAINING_RUN_DIR = (
    OUT_DIR
    / "structured_r1_f1_semantic_targeted_finetuning"
    / "qlora_r8_lambda_0_5_seed_42"
)

# This cache is used ONLY to recover the exact same
# 200 held-out case IDs.
FINE_TUNED_TEST_CACHE_PATH = (
    TRAINING_RUN_DIR
    / "held_out_test_predictions_cache.json"
)


assert FROZEN_F1_CACHE_PATH.exists(), (
    "Frozen F1 cache not found:\n"
    f"{FROZEN_F1_CACHE_PATH}"
)

assert FINE_TUNED_TEST_CACHE_PATH.exists(), (
    "Held-out test cache not found:\n"
    f"{FINE_TUNED_TEST_CACHE_PATH}"
)


# ============================================================
# 2. LOAD SAVED CACHES
# ============================================================

with FROZEN_F1_CACHE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    frozen_cache = json.load(file)

with FINE_TUNED_TEST_CACHE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    held_out_cache = json.load(file)


assert isinstance(
    frozen_cache.get("records"),
    dict,
)

assert isinstance(
    held_out_cache.get("records"),
    dict,
)


print("=" * 100)
print("LOADED SAVED RESULTS")
print("=" * 100)

print(
    "All frozen F1 records:",
    len(frozen_cache["records"]),
)

print(
    "Held-out test case IDs:",
    len(held_out_cache["records"]),
)


# ============================================================
# 3. CONVERT RECORDS TO DATAFRAME
# ============================================================

def records_to_dataframe(records):
    rows = []

    for case_id, record in records.items():
        row = dict(record)

        row["case_id"] = str(
            row.get("case_id", case_id)
        )

        rows.append(row)

    return pd.DataFrame(rows)


frozen_all_df = records_to_dataframe(
    frozen_cache["records"]
)

held_out_case_ids = {
    str(case_id)
    for case_id in held_out_cache["records"].keys()
}


# ============================================================
# 4. KEEP EXACTLY THE SAME 200 TEST CASES
# ============================================================

frozen_f1_df = (
    frozen_all_df[
        frozen_all_df[
            "case_id"
        ].astype(str).isin(
            held_out_case_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(frozen_f1_df) == 200, (
    f"Expected 200 frozen F1 test cases, "
    f"found {len(frozen_f1_df)}."
)

assert set(
    frozen_f1_df["case_id"].astype(str)
) == held_out_case_ids


print(
    "\nLoaded frozen F1 held-out predictions:",
    len(frozen_f1_df),
)


# ============================================================
# 5. NORMALISE COLUMNS
# ============================================================

frozen_f1_df["case_id"] = (
    frozen_f1_df["case_id"]
    .astype(str)
)

frozen_f1_df["case_family"] = (
    frozen_f1_df["case_family"]
    .astype(str)
    .str.lower()
)

frozen_f1_df["gold_label"] = (
    frozen_f1_df["gold_binary_label"]
    .astype(str)
    .str.upper()
)

frozen_f1_df["prediction"] = (
    frozen_f1_df["prediction"]
    .astype(str)
    .str.upper()
)

for column in [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]:
    frozen_f1_df[column] = (
        frozen_f1_df[column]
        .astype(str)
        .str.upper()
    )


frozen_f1_df["correct"] = (
    frozen_f1_df["prediction"]
    ==
    frozen_f1_df["gold_label"]
)


FAMILY_ORDER = [
    "normal",
    "lag",
    "wrong_partner",
    "silent_partner",
]

SEMANTIC_ORDER = [
    "COMPATIBLE",
    "INCOMPATIBLE",
    "LIMITED",
]


# ============================================================
# 6. NUMBER OF CASES PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — TEST CASES PER FAMILY")
print("=" * 100)

display(
    frozen_f1_df[
        "case_family"
    ]
    .value_counts()
    .reindex(
        FAMILY_ORDER,
        fill_value=0,
    )
    .rename("cases")
    .to_frame()
)


# ============================================================
# 7. FINAL PREDICTIONS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — FINAL PREDICTIONS PER FAMILY")
print("=" * 100)

final_predictions_table = pd.crosstab(
    index=frozen_f1_df["case_family"],
    columns=frozen_f1_df["prediction"],
)

final_predictions_table = (
    final_predictions_table.reindex(
        index=FAMILY_ORDER,
        columns=[
            "NORMAL",
            "ANOMALOUS",
        ],
        fill_value=0,
    )
)

final_predictions_table["All"] = (
    final_predictions_table.sum(axis=1)
)

final_predictions_table = pd.concat(
    [
        final_predictions_table,
        pd.DataFrame(
            [
                final_predictions_table.sum(
                    axis=0
                )
            ],
            index=["All"],
        ),
    ]
)

display(final_predictions_table)


# ============================================================
# 8. ACCURACY PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — ACCURACY PER FAMILY")
print("=" * 100)

family_accuracy_df = (
    frozen_f1_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "count",
        ),
        correct_predictions=(
            "correct",
            "sum",
        ),
        accuracy=(
            "correct",
            "mean",
        ),
        predicted_NORMAL=(
            "prediction",
            lambda values: (
                values == "NORMAL"
            ).sum(),
        ),
        predicted_ANOMALOUS=(
            "prediction",
            lambda values: (
                values == "ANOMALOUS"
            ).sum(),
        ),
    )
)

family_accuracy_df["case_family"] = pd.Categorical(
    family_accuracy_df["case_family"],
    categories=FAMILY_ORDER,
    ordered=True,
)

family_accuracy_df = (
    family_accuracy_df
    .sort_values("case_family")
    .reset_index(drop=True)
)

display(family_accuracy_df)


# ============================================================
# 9. SEMANTIC ASSESSMENTS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — SEMANTIC ASSESSMENTS PER FAMILY")
print("=" * 100)

semantic_table = pd.crosstab(
    index=frozen_f1_df["case_family"],
    columns=frozen_f1_df[
        "semantic_assessment"
    ],
)

semantic_table = semantic_table.reindex(
    index=FAMILY_ORDER,
    columns=SEMANTIC_ORDER,
    fill_value=0,
)

semantic_table["All"] = (
    semantic_table.sum(axis=1)
)

semantic_table = pd.concat(
    [
        semantic_table,
        pd.DataFrame(
            [
                semantic_table.sum(
                    axis=0
                )
            ],
            index=["All"],
        ),
    ]
)

display(semantic_table)


# ============================================================
# 10. SEMANTIC PERCENTAGES PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — SEMANTIC PERCENTAGES PER FAMILY")
print("=" * 100)

semantic_percentages = (
    semantic_table
    .loc[
        FAMILY_ORDER,
        SEMANTIC_ORDER,
    ]
    .div(
        semantic_table.loc[
            FAMILY_ORDER,
            "All",
        ],
        axis=0,
    )
    .mul(100)
    .round(2)
)

display(semantic_percentages)


# ============================================================
# 11. PARTICIPATION ASSESSMENTS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — PARTICIPATION ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_df[
            "case_family"
        ],
        columns=frozen_f1_df[
            "participation_assessment"
        ],
        margins=True,
    )
)


# ============================================================
# 12. LOCAL TEMPORAL ASSESSMENTS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — LOCAL TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_df[
            "case_family"
        ],
        columns=frozen_f1_df[
            "local_temporal_assessment"
        ],
        margins=True,
    )
)


# ============================================================
# 13. GLOBAL TEMPORAL ASSESSMENTS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — GLOBAL TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_df[
            "case_family"
        ],
        columns=frozen_f1_df[
            "global_temporal_assessment"
        ],
        margins=True,
    )
)


# ============================================================
# 14. COMBINED TEMPORAL ASSESSMENTS PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — TEMPORAL ASSESSMENTS")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_df[
            "case_family"
        ],
        columns=frozen_f1_df[
            "temporal_assessment"
        ],
        margins=True,
    )
)


# ============================================================
# 15. DECISIVE DIMENSION PER FAMILY
# ============================================================

print("\n" + "=" * 100)
print("FROZEN F1 — DECISIVE DIMENSION")
print("=" * 100)

display(
    pd.crosstab(
        index=frozen_f1_df[
            "case_family"
        ],
        columns=frozen_f1_df[
            "decisive_dimension"
        ],
        margins=True,
    )
)


# ============================================================
# 16. DETAILED RESULTS PER FAMILY
# ============================================================

DETAILED_COLUMNS = [
    "case_id",
    "source_group_id",
    "case_variant",
    "gold_label",
    "prediction",
    "correct",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "schema_exact",
]

for family in FAMILY_ORDER:
    family_df = (
        frozen_f1_df[
            frozen_f1_df[
                "case_family"
            ] == family
        ]
        .copy()
        .sort_values("case_id")
        .reset_index(drop=True)
    )

    print("\n" + "=" * 100)
    print(
        f"FROZEN F1 — "
        f"{family.upper()} CASES"
    )
    print("=" * 100)

    display(
        family_df[
            DETAILED_COLUMNS
        ]
    )


# ============================================================
# 17. WRONG-PARTNER CASES GROUPED BY SEMANTIC RESULT
# ============================================================

wrong_partner_df = (
    frozen_f1_df[
        frozen_f1_df[
            "case_family"
        ] == "wrong_partner"
    ]
    .copy()
)

for semantic_value in SEMANTIC_ORDER:
    subset = (
        wrong_partner_df[
            wrong_partner_df[
                "semantic_assessment"
            ] == semantic_value
        ]
        .sort_values("case_id")
        .reset_index(drop=True)
    )

    print("\n" + "=" * 100)
    print(
        "FROZEN F1 WRONG PARTNER — "
        f"{semantic_value}"
    )
    print("=" * 100)

    print("Cases:", len(subset))

    display(
        subset[
            DETAILED_COLUMNS
        ]
    )


# ============================================================
# 18. SAVE CLEAN CASE-LEVEL TABLE
# ============================================================

OUTPUT_CSV_PATH = (
    TRAINING_RUN_DIR
    / "frozen_f1_clean_case_level_results.csv"
)

frozen_f1_df[
    [
        "case_id",
        "source_group_id",
        "case_family",
        *DETAILED_COLUMNS[2:],
    ]
].to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)


print("\n" + "=" * 100)
print("COMPLETE")
print("=" * 100)

print("Saved clean frozen F1 results to:")
print(OUTPUT_CSV_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOADED SAVED RESULTS
All frozen F1 records: 400
Held-out test case IDs: 200

Loaded frozen F1 held-out predictions: 200

FROZEN F1 — TEST CASES PER FAMILY


,cases
case_family,
normal,50
lag,50
wrong_partner,50
silent_partner,50



FROZEN F1 — FINAL PREDICTIONS PER FAMILY


prediction,NORMAL,ANOMALOUS,All
normal,38,12,50
lag,7,43,50
wrong_partner,4,46,50
silent_partner,0,50,50
All,49,151,200



FROZEN F1 — ACCURACY PER FAMILY


...


FROZEN F1 WRONG PARTNER — INCOMPATIBLE
Cases: 19


,case_id,source_group_id,case_variant,gold_label,prediction,correct,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact
0,consolidation_wrong_partner_000,heldout_source_000,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
1,consolidation_wrong_partner_002,heldout_source_002,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
2,consolidation_wrong_partner_004,heldout_source_004,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
3,consolidation_wrong_partner_020,heldout_source_020,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
4,consolidation_wrong_partner_026,heldout_source_026,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
5,consolidation_wrong_partner_028,heldout_source_028,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
6,consolidation_wrong_partner_029,heldout_source_029,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
7,consolidation_wrong_partner_036,heldout_source_036,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,INCOMPATIBLE,SEMANTIC,True
8,consolidation_wrong_partner_039,heldout_source_039,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True
9,consolidation_wrong_partner_041,heldout_source_041,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,INCOMPATIBLE,TEMPORAL,True



FROZEN F1 WRONG PARTNER — LIMITED
Cases: 12


,case_id,source_group_id,case_variant,gold_label,prediction,correct,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,schema_exact
0,consolidation_wrong_partner_007,heldout_source_007,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
1,consolidation_wrong_partner_010,heldout_source_010,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,LIMITED,SEMANTIC,True
2,consolidation_wrong_partner_032,heldout_source_032,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
3,consolidation_wrong_partner_048,heldout_source_048,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
4,consolidation_wrong_partner_049,heldout_source_049,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
5,consolidation_wrong_partner_062,heldout_source_062,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
6,consolidation_wrong_partner_074,heldout_source_074,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
7,consolidation_wrong_partner_078,heldout_source_078,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True
8,consolidation_wrong_partner_086,heldout_source_086,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,NORMAL,NORMAL,NORMAL,LIMITED,SEMANTIC,True
9,consolidation_wrong_partner_087,heldout_source_087,wrong_partner,ANOMALOUS,ANOMALOUS,True,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,LIMITED,TEMPORAL,True



COMPLETE
Saved clean frozen F1 results to:
/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/structured_r1_f1_semantic_targeted_finetuning/qlora_r8_lambda_0_5_seed_42/frozen_f1_clean_case_level_results.csv
